# High-norm tokens vs attention sinks in FLUX

This notebook tests whether the same image tokens that become **high-norm residual tokens** also act as **attention sinks** in FLUX-style Diffusion Transformers.

The core question is:

> For each selected FLUX block and denoising step, are the highest-residual-norm image tokens also the image tokens that receive the most incoming image-to-image attention?

The notebook produces:

- `head_level_metrics.csv`: one row per block/head with sink token, sink strength, top-k **Jaccard overlap with chance baseline**, tie-aware Spearman correlation **with p-values**, entropy, value/key norms, and **text-attention leak statistics**.
- `token_level_metrics.csv`: one row per block/head/token with attention mass, attention rank, per-stage norm, norm ranks, and high-norm flags.
- `stage_level_metrics.csv`, `transition_metrics.csv`: per-stage norms and sink-growth diagnostics with matched controls.
- visualizations:
  - **spatial token maps**: attention, norm, and overlap categories rendered on the 2D patch grid next to the generated image (`spatial_maps/`)
  - **per-head sink location maps**: do all heads share one sink, or does each head pick its own?
  - **rank-rank hexbins**: do the norm and attention orderings agree across the *whole* distribution, not just the top-k tail? (`rank_rank/`)
  - **Jaccard overlap vs chance**: dashed lines mark the random-set expectation (`overlap_summaries/`)
  - **text-attention leak**: how much image-query mass flows to text tokens, and how often the head's true sink is a text token (`text_attention/`)
  - plus attention heatmaps, norm/attention curves, sink-annotated scatter plots, correlation heatmaps, rank histograms, and token trajectories.

Key methodological notes:

1. **Text-token confound.** In MMDiT joint attention, image queries also attend to text tokens, and T5 padding tokens are prime sink candidates. Renormalized image-to-image attention hides this. The capture now also records the raw mass flowing to each text token (`attn_img2text`), and `head_level_metrics.csv` flags heads whose strongest raw receiver is a text token (`global_sink_is_text`). Interpret image-token sink claims only in light of `text_attention_fraction`.
2. **Chance-corrected overlap.** Binary "any top-k overlap" saturates as k grows. Use `topk_jaccard` columns and compare against `topk_jaccard_chance`.
3. Explicit attention capture materializes full attention probabilities for the selected blocks. Start at `512 x 512`; only move to `1024 x 1024` once the pipeline works.


In [ ]:
# ================================================================
# Cell 1: Install dependencies
# ================================================================
# This notebook is written against Diffusers 0.35.1 FLUX internals.
# Restart runtime after this cell if Colab asks you to.

%pip install -q "diffusers==0.35.1" "transformers==4.56.2" accelerate safetensors sentencepiece protobuf huggingface_hub
%pip install -q numpy pandas matplotlib seaborn scipy tqdm
%pip install lpips

In [ ]:
# ================================================================
# Cell 2: Optional Hugging Face login + output path
# ================================================================
# Uncomment login if your FLUX model access requires a token.

# from huggingface_hub import notebook_login
from huggingface_hub import notebook_login
notebook_login()

# Optional Google Drive persistence.
from google.colab import drive
drive.mount('/content/drive')
PERSISTENT_OUTPUT_DIR = '/content/drive/MyDrive/highnorm_sink_analysis_rewritten'

#PERSISTENT_OUTPUT_DIR = 'outputs/highnorm_sink_analysis_rewritten'

In [ ]:
# ================================================================
# Cell 3: Imports and reproducibility
# ================================================================

from __future__ import annotations

import json
import math
import random
import warnings
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(context="notebook", style="whitegrid")
warnings.filterwarnings("ignore", category=UserWarning)


def seed_everything(seed: int = 0) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(0)
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

In [ ]:
# ================================================================
# Cell 4: Experiment configuration and capture containers
# ================================================================

TEXT_LENGTH = 512


@dataclass
class ExperimentConfig:
    model_name: str = "black-forest-labs/FLUX.1-schnell"
    prompts: List[str] = field(default_factory=lambda: [
        "A close-up portrait of an astronaut in a reflective helmet, cinematic lighting, ultra detailed"
    ])
    seeds: List[int] = field(default_factory=lambda: [42])
    num_inference_steps: int = 4

    # Step indices are 0-based. With 4 steps, [3] means final denoising step.
    capture_timesteps: List[int] = field(default_factory=lambda: [3])

    # FLUX has 19 dual-stream blocks and 38 single-stream blocks in the common config.
    dual_blocks: List[int] = field(default_factory=lambda: [18])
    single_blocks: List[int] = field(default_factory=lambda: [0])

    # Start small because explicit attention capture stores [B, H, N, N].
    height: int = 512
    width: int = 512

    guidance_scale: float = 0.0
    dtype: str = "bfloat16"  # float16, bfloat16, float32
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    text_length: int = TEXT_LENGTH

    topk: List[int] = field(default_factory=lambda: [1, 5, 10])
    mad_z: float = 6.0
    max_heatmap_tokens: int = 512

    output_dir: str = PERSISTENT_OUTPUT_DIR
    save_generated_images: bool = True

    @property
    def grid_h(self) -> int:
        # FLUX: VAE downsamples by 8, then 2x2 latent patchification -> /16 total.
        return self.height // 16

    @property
    def grid_w(self) -> int:
        return self.width // 16


@dataclass
class BlockCapture:
    model_name: str
    prompt_id: int
    prompt: str
    seed: int
    timestep_index: int
    timestep_value: Optional[float]
    block_type: str       # "dual" or "single"
    block_id: int

    # Spatial layout of image tokens (row-major patch grid). Filled by the runner.
    grid_h: Optional[int] = None
    grid_w: Optional[int] = None

    # Residual/activation states are image-token only: [B, N_img, C]
    pre_block: Optional[torch.Tensor] = None
    attention_output: Optional[torch.Tensor] = None
    post_attention_residual: Optional[torch.Tensor] = None  # dual blocks only
    post_block: Optional[torch.Tensor] = None

    # Attention objects.
    attn_img2img: Optional[torch.Tensor] = None             # [B, H, Q_img, K_img], raw (not renormalized)
    attn_img2text: Optional[torch.Tensor] = None            # [B, H, K_text]: mean over image queries of raw mass to each text token
    key_img: Optional[torch.Tensor] = None                  # [B, H, K_img, D]
    value_img: Optional[torch.Tensor] = None                # [B, H, K_img, D]

    @property
    def name(self) -> str:
        return f"p{self.prompt_id}_s{self.seed}_t{self.timestep_index}_{self.block_type}{self.block_id}"

    @property
    def block_name(self) -> str:
        return f"{self.block_type}_block{self.block_id}"

    def grid_shape(self, n_tokens: int) -> Tuple[int, int]:
        """Return (rows, cols) for reshaping token vectors to the patch grid."""
        if self.grid_h and self.grid_w and self.grid_h * self.grid_w == n_tokens:
            return self.grid_h, self.grid_w
        side = int(round(math.sqrt(n_tokens)))
        if side * side == n_tokens:
            return side, side
        return 1, n_tokens  # fall back to a 1-row strip


class CaptureStore:
    def __init__(self, cfg: ExperimentConfig):
        self.cfg = cfg
        self.records: Dict[Tuple[int, int, int, str, int], BlockCapture] = {}
        self.handles: List[Any] = []
        self.prompt_id: int = 0
        self.prompt: str = ""
        self.seed: int = 0
        self.current_timestep_index: int = -1
        self.current_timestep_value: Optional[float] = None

    def want_timestep(self) -> bool:
        return self.current_timestep_index in set(self.cfg.capture_timesteps)

    def record_key(self, block_type: str, block_id: int) -> Tuple[int, int, int, str, int]:
        return (self.prompt_id, self.seed, self.current_timestep_index, block_type, block_id)

    def rec(self, block_type: str, block_id: int) -> BlockCapture:
        k = self.record_key(block_type, block_id)
        if k not in self.records:
            self.records[k] = BlockCapture(
                model_name=self.cfg.model_name,
                prompt_id=self.prompt_id,
                prompt=self.prompt,
                seed=self.seed,
                timestep_index=self.current_timestep_index,
                timestep_value=self.current_timestep_value,
                block_type=block_type,
                block_id=block_id,
                grid_h=self.cfg.grid_h,
                grid_w=self.cfg.grid_w,
            )
        return self.records[k]

    def close(self) -> None:
        for h in self.handles:
            try:
                h.remove()
            except Exception:
                pass
        self.handles.clear()


def detach_cpu(x: torch.Tensor) -> torch.Tensor:
    return x.detach().float().cpu()


def ensure_output_dirs(root: Path) -> None:
    folders = [
        "generated_images",
        "attention_heatmaps",
        "norm_attention_curves",
        "scatter_norm_vs_attention",
        "overlap_summaries",
        "correlation_heatmaps",
        "rank_histograms",
        "token_trajectories",
        "spatial_maps",
        "rank_rank",
        "text_attention",
        "tables",
    ]
    for folder in folders:
        (root / folder).mkdir(parents=True, exist_ok=True)


In [ ]:
# ================================================================
# Cell 5: Metric utilities
# ================================================================

from scipy import stats as sps


def stage_tensors(rec: BlockCapture) -> Dict[str, torch.Tensor]:
    """Return only stages present for this block."""
    out = {}
    if rec.pre_block is not None:
        out["pre_block"] = rec.pre_block
    if rec.attention_output is not None:
        out["attention_output"] = rec.attention_output
    if rec.post_attention_residual is not None:
        out["post_attention_residual"] = rec.post_attention_residual
    if rec.post_block is not None:
        out["post_block"] = rec.post_block
    return out


def rank_desc(values: torch.Tensor) -> torch.Tensor:
    """1 means largest value. Ordinal ranks (no tie handling) — used for rank columns only.
    Correlations use scipy.stats.spearmanr, which handles ties properly."""
    order = torch.argsort(values, descending=True)
    ranks = torch.empty_like(order)
    ranks[order] = torch.arange(1, values.numel() + 1, device=values.device)
    return ranks


def compute_incoming_attention_mass(attn_img2img: torch.Tensor, renormalize: bool = True) -> torch.Tensor:
    """
    Average incoming attention per key token over image queries.

    Args:
        attn_img2img: [B, H, Q_img, K_img] raw probabilities (rows may sum to < 1
            because image queries also attend to text tokens).
        renormalize: if True, renormalize each query row over image keys only, so
            the result answers "which image token wins, conditional on attending to
            the image". If False, keep raw mass, so the result is comparable with
            text-token incoming mass.
    Returns:
        incoming mass: [B, H, K_img]
    """
    if renormalize:
        attn = attn_img2img / attn_img2img.sum(dim=-1, keepdim=True).clamp(min=1e-8)
    else:
        attn = attn_img2img
    return attn.mean(dim=-2)


def identify_sink_tokens(attn_img2img: torch.Tensor, topk: Sequence[int]) -> Dict[str, torch.Tensor]:
    incoming = compute_incoming_attention_mass(attn_img2img, renormalize=True)
    max_k = min(max(topk), incoming.shape[-1])
    vals, idx = torch.topk(incoming, k=max_k, dim=-1)
    return {
        "incoming": incoming,
        "top_values": vals,
        "top_indices": idx,
        "sink_token": idx[..., 0],
        "sink_strength": vals[..., 0],
    }


def compute_highnorm_tokens(norms: torch.Tensor, topk: Sequence[int], mad_z: float = 6.0) -> Dict[str, Any]:
    max_k = min(max(topk), norms.shape[-1])
    vals, idx = torch.topk(norms, k=max_k, dim=-1)
    med = norms.median(dim=-1, keepdim=True).values
    mad = (norms - med).abs().median(dim=-1, keepdim=True).values.clamp(min=1e-8)
    robust_mask = norms > (med + mad_z * 1.4826 * mad)
    return {"top_values": vals, "top_indices": idx, "ranks": rank_desc(norms), "robust_mask": robust_mask}


def spearman_with_p(x: torch.Tensor, y: torch.Tensor) -> Tuple[float, float]:
    """Tie-aware Spearman correlation with p-value via scipy."""
    x = x.float().flatten().numpy()
    y = y.float().flatten().numpy()
    if x.size < 3 or np.allclose(x, x[0]) or np.allclose(y, y[0]):
        return float("nan"), float("nan")
    r, p = sps.spearmanr(x, y)
    return float(r), float(p)


def topk_jaccard(set_a: set, set_b: set) -> float:
    union = set_a | set_b
    if not union:
        return float("nan")
    return len(set_a & set_b) / len(union)


def expected_topk_jaccard(k: int, n: int) -> float:
    """Expected Jaccard of two independent uniform-random k-subsets of n tokens.
    E|A∩B| = k^2/n; J ≈ E|∩| / (2k − E|∩|). Reference chance level for overlap plots."""
    if n <= 0 or k <= 0:
        return float("nan")
    ei = (k * k) / n
    return ei / max(2 * k - ei, 1e-8)


def attention_entropy(attn_img2img: torch.Tensor) -> torch.Tensor:
    attn = attn_img2img / attn_img2img.sum(dim=-1, keepdim=True).clamp(min=1e-8)
    return -(attn.clamp(min=1e-12).log() * attn).sum(dim=-1).mean(dim=-1)  # [B, H]


def pct_attention_to_tokens(incoming: torch.Tensor, token_ids: torch.Tensor) -> float:
    token_ids = token_ids.to(incoming.device)
    return float(incoming[token_ids].sum().item() / incoming.sum().clamp(min=1e-8).item())


def text_attention_stats(rec: BlockCapture) -> Optional[Dict[str, torch.Tensor]]:
    """
    How much of the image queries' attention leaks to text tokens, per head.

    Returns per-head:
      - text_fraction [H]: total raw mass image queries put on text tokens
      - text_sink_token [H], text_sink_strength [H]: strongest text receiver
      - global_sink_is_text [H]: True if the strongest raw receiver over the full
        (text + image) key axis is a text token. If this is often True, the
        "image sink" from renormalized img->img attention is not the head's real sink.
    """
    if rec.attn_img2text is None or rec.attn_img2img is None:
        return None
    text_incoming = rec.attn_img2text[0]                                          # [H, K_text]
    img_incoming_raw = compute_incoming_attention_mass(rec.attn_img2img, renormalize=False)[0]  # [H, K_img]
    text_fraction = text_incoming.sum(dim=-1)                                     # [H]
    text_sink_strength, text_sink_token = text_incoming.max(dim=-1)
    img_max, _ = img_incoming_raw.max(dim=-1)
    return {
        "text_fraction": text_fraction,
        "text_sink_token": text_sink_token,
        "text_sink_strength": text_sink_strength,
        "global_sink_is_text": text_sink_strength > img_max,
    }


def attention_weighted_value_contribution(attn_img2img: torch.Tensor, value_img: torch.Tensor) -> torch.Tensor:
    """
    Per-token contribution proxy = incoming attention mass * value-vector norm.

    Returns [B, H, K_img]. Handles both [B,H,K,D] and [B,K,H,D] value layouts.
    """
    incoming = compute_incoming_attention_mass(attn_img2img)  # [B,H,K]
    value_norm = value_img.norm(dim=-1)
    if value_norm.shape != incoming.shape:
        if value_norm.ndim == 3 and value_norm.shape[1] == incoming.shape[2] and value_norm.shape[2] == incoming.shape[1]:
            value_norm = value_norm.transpose(1, 2)
    if value_norm.shape != incoming.shape:
        raise RuntimeError(f"value_norm shape {tuple(value_norm.shape)} does not match incoming {tuple(incoming.shape)}")
    return incoming * value_norm


def validate_capture(rec: BlockCapture) -> None:
    stages = stage_tensors(rec)
    if "pre_block" not in stages or "post_block" not in stages or "attention_output" not in stages:
        raise AssertionError(f"Missing core stages for {rec.name}: {list(stages.keys())}")
    if rec.attn_img2img is None:
        raise AssertionError(f"Missing attention for {rec.name}")
    if rec.key_img is None or rec.value_img is None:
        raise AssertionError(f"Missing key/value for {rec.name}")

    n = stages["pre_block"].shape[1]
    if rec.attn_img2img.shape[-1] != n or rec.attn_img2img.shape[-2] != n:
        raise AssertionError(f"Token mismatch in {rec.name}: pre has {n}, attn has {tuple(rec.attn_img2img.shape)}")

    for name, tensor in {**stages, "attn_img2img": rec.attn_img2img, "key_img": rec.key_img, "value_img": rec.value_img}.items():
        if not torch.isfinite(tensor).all():
            raise AssertionError(f"NaN/Inf in {rec.name}:{name}")

    row_sums = (rec.attn_img2img / rec.attn_img2img.sum(dim=-1, keepdim=True).clamp(min=1e-8)).sum(dim=-1)
    if not torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-4):
        raise AssertionError(f"Renormalization failed for {rec.name}")

    # If text attention was captured, raw img mass + text mass should be close to 1 per query row on average.
    if rec.attn_img2text is not None:
        total = compute_incoming_attention_mass(rec.attn_img2img, renormalize=False)[0].sum(-1) + rec.attn_img2text[0].sum(-1)
        if not torch.allclose(total, torch.ones_like(total), atol=2e-3):
            raise AssertionError(f"img+text attention mass != 1 for {rec.name}: {total}")


In [ ]:
# ================================================================
# Cell 6: Build CSV metric tables
# ================================================================

def build_metric_tables(cfg: ExperimentConfig, records: Mapping[Any, BlockCapture]) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    head_rows: List[dict] = []
    token_rows: List[dict] = []
    stage_rows: List[dict] = []

    for rec in records.values():
        validate_capture(rec)
        stages = stage_tensors(rec)
        norms_by_stage = {name: tensor[0].norm(dim=-1) for name, tensor in stages.items()}
        highnorm_by_stage = {name: compute_highnorm_tokens(norms, cfg.topk, cfg.mad_z) for name, norms in norms_by_stage.items()}

        sinks = identify_sink_tokens(rec.attn_img2img, cfg.topk)
        incoming_all = sinks["incoming"][0]  # [H, N]
        entropy_all = attention_entropy(rec.attn_img2img)[0]
        key_norm = rec.key_img[0].norm(dim=-1)
        value_norm = rec.value_img[0].norm(dim=-1)
        value_contrib = attention_weighted_value_contribution(rec.attn_img2img, rec.value_img)[0]
        text_stats = text_attention_stats(rec)

        H, N = incoming_all.shape
        base = dict(
            model_name=rec.model_name,
            prompt_id=rec.prompt_id,
            prompt=rec.prompt,
            seed=rec.seed,
            timestep_index=rec.timestep_index,
            timestep_value=rec.timestep_value,
            block_type=rec.block_type,
            block_id=rec.block_id,
            block_name=rec.block_name,
            num_image_tokens=N,
        )

        # Stage-level rows, independent of head.
        for stage, norms in norms_by_stage.items():
            hn = highnorm_by_stage[stage]
            for tok in range(N):
                stage_rows.append({
                    **base,
                    "stage": stage,
                    "token_id": tok,
                    "norm": float(norms[tok]),
                    "norm_rank": int(hn["ranks"][tok]),
                    "is_mad_outlier": bool(hn["robust_mask"][tok]),
                    **{f"is_highnorm_top{k}": bool(tok in set(map(int, hn["top_indices"][:min(k, N)].tolist()))) for k in cfg.topk},
                })

        for h in range(H):
            incoming = incoming_all[h]
            incoming_rank = rank_desc(incoming)
            sink_token = int(sinks["sink_token"][0, h])
            sink_top_sets = {
                k: set(map(int, sinks["top_indices"][0, h, :min(k, N)].tolist())) for k in cfg.topk
            }

            hrow = {
                **base,
                "head_id": h,
                "sink_token_id": sink_token,
                "sink_strength": float(sinks["sink_strength"][0, h]),
                "attention_entropy": float(entropy_all[h]),
                "key_norm_sink": float(key_norm[h, sink_token]),
                "value_norm_sink": float(value_norm[h, sink_token]),
                "value_contribution_sink": float(value_contrib[h, sink_token]),
            }

            if text_stats is not None:
                hrow["text_attention_fraction"] = float(text_stats["text_fraction"][h])
                hrow["text_sink_token_id"] = int(text_stats["text_sink_token"][h])
                hrow["text_sink_strength"] = float(text_stats["text_sink_strength"][h])
                hrow["global_sink_is_text"] = bool(text_stats["global_sink_is_text"][h])

            for stage, norms in norms_by_stage.items():
                hn = highnorm_by_stage[stage]
                top_ids = hn["top_indices"]
                top1 = int(top_ids[0])
                r, p = spearman_with_p(norms, incoming)
                hrow[f"top1_norm_token__{stage}"] = top1
                hrow[f"sink_rank_by_norm__{stage}"] = int(hn["ranks"][sink_token])
                hrow[f"spearman_norm_vs_incoming__{stage}"] = r
                hrow[f"spearman_pvalue__{stage}"] = p
                for k in cfg.topk:
                    kk = min(k, N)
                    high_set = set(map(int, top_ids[:kk].tolist()))
                    hrow[f"top{k}_overlap__{stage}"] = int(len(high_set & sink_top_sets[k]) > 0)
                    hrow[f"top{k}_jaccard__{stage}"] = topk_jaccard(high_set, sink_top_sets[k])
                    hrow[f"top{k}_jaccard_chance"] = expected_topk_jaccard(kk, N)
                    hrow[f"pct_attention_mass_to_top{k}_highnorm__{stage}"] = pct_attention_to_tokens(incoming, top_ids[:kk])

            head_rows.append(hrow)

            for tok in range(N):
                trow = {
                    **base,
                    "head_id": h,
                    "token_id": tok,
                    "incoming_attention_mass": float(incoming[tok]),
                    "incoming_attention_rank": int(incoming_rank[tok]),
                    "is_sink_top1": bool(tok == sink_token),
                    **{f"is_sink_top{k}": bool(tok in sink_top_sets[k]) for k in cfg.topk},
                    "key_norm": float(key_norm[h, tok]),
                    "value_norm": float(value_norm[h, tok]),
                    "value_contribution": float(value_contrib[h, tok]),
                }

                for stage, norms in norms_by_stage.items():
                    hn = highnorm_by_stage[stage]
                    trow[f"norm__{stage}"] = float(norms[tok])
                    trow[f"norm_rank__{stage}"] = int(hn["ranks"][tok])
                    trow[f"is_mad_outlier__{stage}"] = bool(hn["robust_mask"][tok])
                    top_ids = hn["top_indices"]
                    for k in cfg.topk:
                        kk = min(k, N)
                        trow[f"is_highnorm_top{k}__{stage}"] = bool(tok in set(map(int, top_ids[:kk].tolist())))

                token_rows.append(trow)

    return pd.DataFrame(head_rows), pd.DataFrame(token_rows), pd.DataFrame(stage_rows)


def transition_metrics(cfg: ExperimentConfig, token_df: pd.DataFrame) -> pd.DataFrame:
    """Simple sink-growth and later-sink tracking diagnostics."""
    rows = []
    if token_df.empty:
        return pd.DataFrame(rows)

    available_post_cols = [c for c in token_df.columns if c.startswith("norm__")]
    if "norm__pre_block" not in available_post_cols or "norm__post_block" not in available_post_cols:
        return pd.DataFrame(rows)

    group_cols = ["model_name", "prompt_id", "seed", "timestep_index", "block_type", "block_id", "head_id"]

    for keys, g in token_df.groupby(group_cols):
        sink_rows = g[g["is_sink_top1"]]
        if sink_rows.empty:
            continue
        sink = sink_rows.iloc[0]
        nons = g[~g["is_sink_top1"]].copy()
        if nons.empty:
            continue
        nons["pre_dist"] = (nons["norm__pre_block"] - sink["norm__pre_block"]).abs()
        ctrl = nons.sort_values("pre_dist").iloc[0]

        sink_growth = sink["norm__post_block"] - sink["norm__pre_block"]
        ctrl_growth = ctrl["norm__post_block"] - ctrl["norm__pre_block"]
        nons_growth = nons["norm__post_block"] - nons["norm__pre_block"]
        nons_growth_std = nons_growth.std()

        rows.append(dict(
            **dict(zip(group_cols, keys)),
            sink_token_id=int(sink["token_id"]),
            control_token_id=int(ctrl["token_id"]),
            sink_norm_growth=float(sink_growth),
            control_norm_growth=float(ctrl_growth),
            paired_difference=float(sink_growth - ctrl_growth),
            effect_size=float((sink_growth - ctrl_growth) / nons_growth_std) if nons_growth_std and np.isfinite(nons_growth_std) else np.nan,
            sink_growth_percentile=float((nons_growth < sink_growth).mean()),
            sink_grows_more=bool(sink_growth > ctrl_growth),
            control_type="nearest_pre_norm_non_sink",
        ))

    # Track top post-block high-norm tokens in dual18 into single0/1 attention ranks when available.
    base_cols = ["model_name", "prompt_id", "seed", "timestep_index"]
    for keys, g in token_df.groupby(base_cols):
        src = g[(g["block_type"] == "dual") & (g["block_id"] == 18) & (g["head_id"] == g["head_id"].min())]
        tgt = g[(g["block_type"] == "single") & (g["block_id"].isin([0, 1]))]
        if src.empty or tgt.empty or "norm_rank__post_block" not in src.columns:
            continue
        top_tokens = src.sort_values("norm_rank__post_block").head(10)["token_id"].astype(int).tolist()
        for tok in top_tokens:
            tt = tgt[tgt["token_id"] == tok]
            if tt.empty:
                continue
            rows.append(dict(
                **dict(zip(base_cols, keys)),
                source_block="dual_block18",
                target_block="single_block0/1",
                token_id=int(tok),
                source_post_block_norm_rank=int(src[src["token_id"] == tok]["norm_rank__post_block"].iloc[0]),
                target_best_incoming_attention_rank=int(tt["incoming_attention_rank"].min()),
                target_num_heads_top1_sink=int(tt["is_sink_top1"].sum()),
                target_num_heads_top5_sink=int(tt["is_sink_top5"].sum()) if "is_sink_top5" in tt.columns else np.nan,
                control_type="dual18_highnorm_to_single_sink_tracking",
            ))

    return pd.DataFrame(rows)


In [ ]:
# ================================================================
# Cell 7: Visualization functions
# ================================================================

CATEGORY_COLORS = {
    "ordinary": "#d9d9d9",
    "high-norm only": "#4c72b0",
    "sink only": "#dd8452",
    "overlap": "#c44e52",
}


def save_attention_heatmaps(cfg: ExperimentConfig, records: Mapping[Any, BlockCapture]) -> None:
    root = Path(cfg.output_dir)
    ensure_output_dirs(root)
    for rec in records.values():
        attn = rec.attn_img2img[0]
        attn = attn / attn.sum(dim=-1, keepdim=True).clamp(min=1e-8)
        attn_np = attn.numpy()
        H, N, _ = attn_np.shape
        step = max(1, N // cfg.max_heatmap_tokens)
        cols = min(4, H)
        rows = math.ceil(H / cols)
        fig, axes = plt.subplots(rows, cols, figsize=(4.2 * cols, 3.6 * rows), squeeze=False)
        for h in range(rows * cols):
            ax = axes[h // cols][h % cols]
            if h >= H:
                ax.axis("off")
                continue
            im = ax.imshow(attn_np[h, ::step, ::step], aspect="auto", origin="lower")
            ax.set_title(f"{rec.block_name} head {h}")
            ax.set_xlabel("key image token")
            ax.set_ylabel("query image token")
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        fig.suptitle(f"Image-to-image attention — {rec.name}", y=1.02)
        fig.tight_layout()
        fig.savefig(root / "attention_heatmaps" / f"{rec.name}.png", dpi=170, bbox_inches="tight")
        plt.close(fig)


def save_spatial_token_maps(cfg: ExperimentConfig, records: Mapping[Any, BlockCapture]) -> None:
    """
    Flagship figure. For each capture, map token metrics back onto the 2D patch grid:
      [generated image] | incoming attention (head-mean) | post_block norm | category map
    The category map colors each patch: overlap / high-norm only / sink only / ordinary
    using top-10 sets (head-mean attention vs post_block norm).
    """
    from matplotlib.colors import ListedColormap
    from matplotlib.patches import Patch

    root = Path(cfg.output_dir)
    ensure_output_dirs(root)
    k_cat = 10

    for rec in records.values():
        stages = stage_tensors(rec)
        if "post_block" not in stages or rec.attn_img2img is None:
            continue
        norms = stages["post_block"][0].norm(dim=-1)            # [N]
        N = norms.numel()
        gh, gw = rec.grid_shape(N)

        incoming_heads = compute_incoming_attention_mass(rec.attn_img2img)[0]  # [H, N]
        incoming_mean = incoming_heads.mean(dim=0)                              # [N]

        kk = min(k_cat, N)
        high_set = set(torch.topk(norms, kk).indices.tolist())
        sink_set = set(torch.topk(incoming_mean, kk).indices.tolist())
        cat = np.zeros(N, dtype=int)  # 0 ordinary, 1 high-norm only, 2 sink only, 3 overlap
        for t in high_set:
            cat[t] = 1
        for t in sink_set:
            cat[t] = 3 if cat[t] == 1 else 2

        img_path = root / "generated_images" / f"prompt{rec.prompt_id}_seed{rec.seed}.png"
        have_img = img_path.exists()
        ncols = 4 if have_img else 3
        fig, axes = plt.subplots(1, ncols, figsize=(4.4 * ncols, 4.4))
        ax_i = 0
        if have_img:
            from PIL import Image
            axes[ax_i].imshow(Image.open(img_path))
            axes[ax_i].set_title("generated image")
            axes[ax_i].axis("off")
            ax_i += 1

        im0 = axes[ax_i].imshow(incoming_mean.reshape(gh, gw).numpy(), cmap="viridis")
        axes[ax_i].set_title("incoming attention (head mean)")
        axes[ax_i].axis("off")
        fig.colorbar(im0, ax=axes[ax_i], fraction=0.046, pad=0.04)
        ax_i += 1

        im1 = axes[ax_i].imshow(norms.reshape(gh, gw).numpy(), cmap="magma")
        axes[ax_i].set_title("post-block token norm")
        axes[ax_i].axis("off")
        fig.colorbar(im1, ax=axes[ax_i], fraction=0.046, pad=0.04)
        ax_i += 1

        cmap = ListedColormap([CATEGORY_COLORS[c] for c in ["ordinary", "high-norm only", "sink only", "overlap"]])
        axes[ax_i].imshow(cat.reshape(gh, gw), cmap=cmap, vmin=0, vmax=3)
        axes[ax_i].set_title(f"top-{kk} category map")
        axes[ax_i].axis("off")
        axes[ax_i].legend(
            handles=[Patch(color=CATEGORY_COLORS[c], label=c) for c in ["high-norm only", "sink only", "overlap"]],
            loc="upper center", bbox_to_anchor=(0.5, -0.03), ncol=3, fontsize=8, frameon=False,
        )

        fig.suptitle(f"Where do sinks and high-norm tokens live? — {rec.name}", y=1.02)
        fig.tight_layout()
        fig.savefig(root / "spatial_maps" / f"{rec.name}.png", dpi=170, bbox_inches="tight")
        plt.close(fig)

        # Per-head sink location map: mark each head's top-1 sink on the grid.
        sink_tokens = torch.argmax(incoming_heads, dim=-1).numpy()  # [H]
        counts = np.zeros(N)
        for t in sink_tokens:
            counts[t] += 1
        plt.figure(figsize=(5.2, 4.6))
        plt.imshow(counts.reshape(gh, gw), cmap="inferno")
        plt.colorbar(label="# heads with top-1 sink here")
        plt.title(f"Per-head sink locations — {rec.name}\n(unique sinks: {len(set(sink_tokens.tolist()))}/{len(sink_tokens)} heads)")
        plt.axis("off")
        plt.tight_layout()
        plt.savefig(root / "spatial_maps" / f"{rec.name}_sink_locations.png", dpi=170)
        plt.close()


def save_rank_rank_histograms(cfg: ExperimentConfig, token_df: pd.DataFrame) -> None:
    """2D histogram of norm rank vs incoming-attention rank, pooled over heads.
    A diagonal ridge = the two orderings agree across the whole distribution,
    not just in the top-k tail."""
    from matplotlib.colors import LogNorm

    root = Path(cfg.output_dir)
    ensure_output_dirs(root)
    norm_rank_cols = [c for c in token_df.columns if c.startswith("norm_rank__")]
    for (btype, bid), g in token_df.groupby(["block_type", "block_id"]):
        n = int(g["num_image_tokens"].iloc[0])
        fig, axes = plt.subplots(1, len(norm_rank_cols), figsize=(4.6 * len(norm_rank_cols), 4.2), squeeze=False)
        for ax, col in zip(axes[0], norm_rank_cols):
            stage = col.replace("norm_rank__", "")
            g_stage = g.dropna(subset=[col])
            if g_stage.empty:
                ax.axis("off")
                ax.set_title(f"{stage} (n/a for this block)")
                continue
            hb = ax.hexbin(g_stage[col], g_stage["incoming_attention_rank"], gridsize=40, mincnt=1, norm=LogNorm(), cmap="viridis")
            ax.plot([1, n], [1, n], "r--", linewidth=1, alpha=0.7)
            ax.set_xlabel(f"norm rank ({stage})")
            ax.set_ylabel("incoming attention rank")
            ax.set_title(stage)
            fig.colorbar(hb, ax=ax, fraction=0.046, pad=0.04)
        fig.suptitle(f"Rank agreement, all heads pooled — {btype}{bid} (1 = largest)", y=1.03)
        fig.tight_layout()
        fig.savefig(root / "rank_rank" / f"rank_rank_{btype}{bid}.png", dpi=170, bbox_inches="tight")
        plt.close(fig)


def save_text_attention_plot(cfg: ExperimentConfig, head_df: pd.DataFrame) -> None:
    """How much image-query attention leaks to text tokens, and how often the true
    (raw) sink is a text token rather than an image token."""
    root = Path(cfg.output_dir)
    ensure_output_dirs(root)
    if "text_attention_fraction" not in head_df.columns:
        return
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    sns.barplot(data=head_df, x="head_id", y="text_attention_fraction", hue="block_name", ax=axes[0])
    axes[0].axhline(0.5, linestyle="--", color="gray", linewidth=1)
    axes[0].set_title("Fraction of image-query attention going to TEXT tokens")
    axes[0].set_ylabel("raw mass to text")
    if "global_sink_is_text" in head_df.columns:
        rate = head_df.groupby("block_name", as_index=False)["global_sink_is_text"].mean()
        sns.barplot(data=rate, x="block_name", y="global_sink_is_text", ax=axes[1])
        axes[1].set_ylim(0, 1)
        axes[1].set_title("Fraction of heads whose strongest raw receiver is a TEXT token")
        axes[1].set_ylabel("rate")
    fig.tight_layout()
    fig.savefig(root / "text_attention" / "text_attention_summary.png", dpi=180)
    plt.close(fig)


def save_norm_attention_curves(cfg: ExperimentConfig, token_df: pd.DataFrame) -> None:
    root = Path(cfg.output_dir)
    ensure_output_dirs(root)
    norm_cols = [c for c in token_df.columns if c.startswith("norm__")]
    group_cols = ["prompt_id", "seed", "timestep_index", "block_type", "block_id", "head_id"]
    for keys, g in token_df.groupby(group_cols):
        # To keep figures manageable, save curves for head 0 only.
        if keys[-1] != 0:
            continue
        g = g.sort_values("token_id")
        fig, ax1 = plt.subplots(figsize=(12, 4))
        ax1.plot(g["token_id"], g["incoming_attention_mass"], label="incoming attention", linewidth=1.5)
        ax1.set_xlabel("image token id")
        ax1.set_ylabel("incoming attention mass")
        ax2 = ax1.twinx()
        for col in norm_cols:
            ax2.plot(g["token_id"], g[col], label=col.replace("norm__", "norm: "), alpha=0.75, linewidth=1)
        ax2.set_ylabel("residual/activation norm")
        lines1, labels1 = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right", fontsize=8)
        title = f"Token curves — p{keys[0]} s{keys[1]} t{keys[2]} {keys[3]}{keys[4]} head{keys[5]}"
        ax1.set_title(title)
        fig.tight_layout()
        fig.savefig(root / "norm_attention_curves" / f"{title.replace(' ', '_')}.png", dpi=170)
        plt.close(fig)


def save_scatter_norm_vs_attention(cfg: ExperimentConfig, token_df: pd.DataFrame) -> None:
    root = Path(cfg.output_dir)
    ensure_output_dirs(root)
    norm_cols = [c for c in token_df.columns if c.startswith("norm__")]
    group_cols = ["prompt_id", "seed", "timestep_index", "block_type", "block_id", "head_id"]
    for keys, g in token_df.groupby(group_cols):
        if keys[-1] != 0:
            continue
        for col in norm_cols:
            stage = col.replace("norm__", "")
            high_col = f"is_highnorm_top5__{stage}"
            sink_col = "is_sink_top5" if "is_sink_top5" in g.columns else "is_sink_top1"
            cls = np.where(
                g[high_col] & g[sink_col], "overlap",
                np.where(g[high_col], "high-norm only", np.where(g[sink_col], "sink only", "ordinary")),
            )
            plt.figure(figsize=(6, 5))
            sns.scatterplot(
                data=g, x=col, y="incoming_attention_mass", hue=cls,
                hue_order=["ordinary", "high-norm only", "sink only", "overlap"],
                palette=CATEGORY_COLORS, s=18, linewidth=0,
            )
            # Annotate the top-1 sink token explicitly.
            sink_row = g[g["is_sink_top1"]]
            if not sink_row.empty:
                plt.scatter(sink_row[col], sink_row["incoming_attention_mass"], marker="*", s=220,
                            facecolor="none", edgecolor="black", linewidth=1.2, zorder=5,
                            label=f"top-1 sink (token {int(sink_row['token_id'].iloc[0])})")
            plt.yscale("log")
            plt.legend(fontsize=8)
            plt.title(f"Norm vs incoming attention — {stage}\n{keys[3]}{keys[4]} head{keys[5]}")
            plt.xlabel(f"{stage} norm")
            plt.ylabel("incoming attention mass (log)")
            plt.tight_layout()
            fname = f"p{keys[0]}_s{keys[1]}_t{keys[2]}_{keys[3]}{keys[4]}_h{keys[5]}_{stage}.png"
            plt.savefig(root / "scatter_norm_vs_attention" / fname, dpi=170)
            plt.close()


def save_overlap_summary(cfg: ExperimentConfig, head_df: pd.DataFrame) -> None:
    """Jaccard overlap of top-k high-norm and top-k sink sets, with the chance level
    for random sets drawn as a dashed line. Bars above the line = real coupling."""
    root = Path(cfg.output_dir)
    ensure_output_dirs(root)
    jac_cols = [c for c in head_df.columns if "_jaccard__" in c]
    if not jac_cols:
        return
    odf = head_df.melt(
        id_vars=["block_type", "block_id", "head_id"],
        value_vars=jac_cols,
        var_name="metric",
        value_name="jaccard",
    )
    odf["topk"] = odf["metric"].str.extract(r"top(\d+)_jaccard").astype(int)
    odf["stage"] = odf["metric"].str.split("__").str[-1]

    n_tokens = int(head_df["num_image_tokens"].iloc[0])
    fig, ax = plt.subplots(figsize=(12, 5))
    sns.barplot(data=odf, x="stage", y="jaccard", hue="topk", errorbar=("ci", 95), ax=ax)
    for i, k in enumerate(sorted(odf["topk"].unique())):
        chance = expected_topk_jaccard(min(k, n_tokens), n_tokens)
        ax.axhline(chance, linestyle="--", linewidth=1, color=sns.color_palette()[i], alpha=0.9)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Jaccard(high-norm top-k, sink top-k)")
    ax.set_title("High-norm ∩ sink overlap vs chance (dashed = random-set expectation)")
    plt.setp(ax.get_xticklabels(), rotation=25, ha="right")
    fig.tight_layout()
    fig.savefig(root / "overlap_summaries" / "topk_jaccard_by_stage.png", dpi=180)
    plt.close(fig)

    # Legacy binary "any overlap" plot, kept for continuity with earlier runs.
    overlap_cols = [c for c in head_df.columns if c.startswith("top") and "_overlap__" in c]
    if overlap_cols:
        odf2 = head_df.melt(
            id_vars=["block_type", "block_id", "head_id"],
            value_vars=overlap_cols,
            var_name="metric",
            value_name="overlap",
        )
        odf2["topk"] = odf2["metric"].str.extract(r"top(\d+)_overlap").astype(int)
        odf2["stage"] = odf2["metric"].str.split("__").str[-1]
        summary = odf2.groupby(["block_type", "block_id", "stage", "topk"], as_index=False)["overlap"].mean()
        plt.figure(figsize=(12, 5))
        sns.barplot(data=summary, x="stage", y="overlap", hue="topk")
        plt.ylim(0, 1)
        plt.xticks(rotation=25, ha="right")
        plt.ylabel("fraction of heads with any overlap")
        plt.title("High-norm top-k ∩ sink top-k (binary, legacy)")
        plt.tight_layout()
        plt.savefig(root / "overlap_summaries" / "topk_overlap_by_stage.png", dpi=180)
        plt.close()


def save_correlation_heatmaps(cfg: ExperimentConfig, head_df: pd.DataFrame) -> None:
    root = Path(cfg.output_dir)
    ensure_output_dirs(root)
    corr_cols = [c for c in head_df.columns if c.startswith("spearman_norm_vs_incoming__")]
    if not corr_cols:
        return
    long = head_df.melt(
        id_vars=["block_type", "block_id", "head_id"],
        value_vars=corr_cols,
        var_name="stage",
        value_name="spearman",
    )
    long["stage"] = long["stage"].str.replace("spearman_norm_vs_incoming__", "", regex=False)
    long["block"] = long["block_type"] + long["block_id"].astype(str)
    for stage, g in long.groupby("stage"):
        pivot = g.pivot_table(index="block", columns="head_id", values="spearman", aggfunc="mean")
        plt.figure(figsize=(max(6, 0.45 * len(pivot.columns)), max(2.5, 0.5 * len(pivot))))
        sns.heatmap(pivot, annot=False, center=0, cmap="coolwarm", vmin=-1, vmax=1)
        plt.title(f"Spearman(norm, incoming attention) — {stage}")
        plt.xlabel("head")
        plt.ylabel("block")
        plt.tight_layout()
        plt.savefig(root / "correlation_heatmaps" / f"spearman_{stage}.png", dpi=180)
        plt.close()


def save_rank_histograms(cfg: ExperimentConfig, head_df: pd.DataFrame) -> None:
    root = Path(cfg.output_dir)
    ensure_output_dirs(root)
    rank_cols = [c for c in head_df.columns if c.startswith("sink_rank_by_norm__")]
    if not rank_cols:
        return
    rdf = head_df.melt(value_vars=rank_cols, var_name="stage", value_name="sink_norm_rank")
    rdf["stage"] = rdf["stage"].str.replace("sink_rank_by_norm__", "", regex=False)
    plt.figure(figsize=(9, 5))
    sns.histplot(data=rdf, x="sink_norm_rank", hue="stage", element="step", stat="count", common_norm=False, bins=30)
    plt.xlabel("rank of sink token by norm; 1 = highest norm")
    plt.title("Are sink tokens also high-norm tokens?")
    plt.tight_layout()
    plt.savefig(root / "rank_histograms" / "sink_rank_by_norm_histogram.png", dpi=180)
    plt.close()


def save_token_trajectories(cfg: ExperimentConfig, stage_df: pd.DataFrame, token_df: pd.DataFrame) -> None:
    root = Path(cfg.output_dir)
    ensure_output_dirs(root)
    if stage_df.empty or token_df.empty:
        return

    # Pick a small set of interesting tokens: union of sinks and top post_block high-norm tokens.
    interesting = set(token_df[token_df["is_sink_top1"]]["token_id"].astype(int).head(20).tolist())
    if "stage" in stage_df.columns:
        top_post = stage_df[stage_df["stage"] == "post_block"].sort_values("norm_rank").head(20)["token_id"].astype(int).tolist()
        interesting.update(top_post)
    interesting = sorted(list(interesting))[:20]
    if not interesting:
        return

    sdf = stage_df[stage_df["token_id"].isin(interesting)].copy()
    sdf["block_order"] = np.where(sdf["block_type"] == "dual", sdf["block_id"], 100 + sdf["block_id"])
    sdf["x"] = sdf["block_type"] + sdf["block_id"].astype(str) + ":" + sdf["stage"]
    order = sdf.sort_values(["block_order", "stage"])["x"].drop_duplicates().tolist()

    plt.figure(figsize=(max(10, 0.45 * len(order)), 5))
    sns.lineplot(data=sdf, x="x", y="norm", hue="token_id", marker="o", estimator=None, alpha=0.8)
    plt.xticks(rotation=45, ha="right")
    plt.title("Norm trajectories for sink/high-norm candidate tokens")
    plt.xlabel("block:stage")
    plt.ylabel("token norm")
    plt.tight_layout()
    plt.savefig(root / "token_trajectories" / "candidate_token_norm_trajectories.png", dpi=180)
    plt.close()


def plot_all_outputs(cfg: ExperimentConfig, records: Mapping[Any, BlockCapture], head_df: pd.DataFrame, token_df: pd.DataFrame, stage_df: pd.DataFrame) -> None:
    save_attention_heatmaps(cfg, records)
    save_spatial_token_maps(cfg, records)
    save_rank_rank_histograms(cfg, token_df)
    save_text_attention_plot(cfg, head_df)
    save_norm_attention_curves(cfg, token_df)
    save_scatter_norm_vs_attention(cfg, token_df)
    save_overlap_summary(cfg, head_df)
    save_correlation_heatmaps(cfg, head_df)
    save_rank_histograms(cfg, head_df)
    save_token_trajectories(cfg, stage_df, token_df)


In [ ]:
# ================================================================
# Cell 8: Synthetic smoke test
# ================================================================
# This verifies metrics and plots without downloading FLUX.
# It now also fakes text-token attention leak and a padding-like text sink,
# and lays tokens on an 8x16 grid so spatial maps can be smoke-tested.


def run_synthetic(cfg: ExperimentConfig) -> Dict[Any, BlockCapture]:
    seed_everything(0)
    records: Dict[Any, BlockCapture] = {}

    for block_type, block_ids in [("dual", cfg.dual_blocks), ("single", cfg.single_blocks)]:
        for block_id in block_ids:
            B, H, N, C, D = 1, 4, 128, 64, 16
            K_TEXT = 64
            TEXT_FRACTION = 0.3
            rec = BlockCapture(
                model_name="synthetic_flux_like",
                prompt_id=0,
                prompt=cfg.prompts[0],
                seed=cfg.seeds[0],
                timestep_index=cfg.capture_timesteps[0],
                timestep_value=0.0,
                block_type=block_type,
                block_id=block_id,
                grid_h=8,
                grid_w=16,
            )

            pre = torch.randn(B, N, C)
            attn_out = 0.05 * torch.randn(B, N, C)
            post_attention = pre + attn_out
            post_block = post_attention + 0.05 * torch.randn(B, N, C)

            # Make a known token both high-norm and, for single blocks, a strong sink.
            sink_token = 31 if block_type == "dual" else 7
            highnorm_token = sink_token if block_type == "single" else 12
            post_block[:, highnorm_token, :] += 8.0
            attn_out[:, sink_token, :] += 3.0

            attn = torch.rand(B, H, N, N) * 0.01
            if block_type == "single":
                attn[:, :, :, sink_token] += 0.25
            else:
                attn[:, :, :, sink_token] += 0.04
            # Raw img->img rows sum to (1 - TEXT_FRACTION); the rest leaks to text.
            attn = attn / attn.sum(dim=-1, keepdim=True) * (1.0 - TEXT_FRACTION)

            # Text incoming mass: mostly on text token 0 (padding-sink-like).
            text_incoming = torch.rand(B, H, K_TEXT) * 0.001
            text_incoming[:, :, 0] += 1.0
            text_incoming = text_incoming / text_incoming.sum(dim=-1, keepdim=True) * TEXT_FRACTION

            rec.pre_block = pre
            rec.attention_output = attn_out
            rec.post_attention_residual = post_attention if block_type == "dual" else None
            rec.post_block = post_block
            rec.attn_img2img = attn
            rec.attn_img2text = text_incoming
            rec.key_img = torch.randn(B, H, N, D)
            rec.value_img = torch.randn(B, H, N, D)

            records[(0, cfg.seeds[0], cfg.capture_timesteps[0], block_type, block_id)] = rec

    return records


cfg = ExperimentConfig(
    model_name="synthetic_flux_like",
    output_dir=f"{PERSISTENT_OUTPUT_DIR}_synthetic",
    dual_blocks=[18],
    single_blocks=[0],
)
root = Path(cfg.output_dir)
root.mkdir(parents=True, exist_ok=True)
ensure_output_dirs(root)
(root / "config.json").write_text(json.dumps(asdict(cfg), indent=2))

records = run_synthetic(cfg)
head_df, token_df, stage_df = build_metric_tables(cfg, records)
trans_df = transition_metrics(cfg, token_df)

head_df.to_csv(root / "head_level_metrics.csv", index=False)
token_df.to_csv(root / "token_level_metrics.csv", index=False)
stage_df.to_csv(root / "stage_level_metrics.csv", index=False)
trans_df.to_csv(root / "transition_metrics.csv", index=False)
plot_all_outputs(cfg, records, head_df, token_df, stage_df)

print(f"Synthetic test wrote outputs to: {root}")
display(head_df.head())


In [ ]:
# ================================================================
# Cell 9: FLUX attention capture processor and runner
# ================================================================
# This is the main implementation cell.
# It replaces SDPA only for the selected blocks so we can explicitly save attention probabilities.

from diffusers.models.transformers.transformer_flux import apply_rotary_emb


def _get_qkv_projections_for_flux(attn, hidden_states, encoder_hidden_states=None):
    """Compatible with Diffusers 0.35.1 FluxAttention."""
    if getattr(attn, "fused_projections", False) and hasattr(attn, "to_qkv"):
        query, key, value = attn.to_qkv(hidden_states).chunk(3, dim=-1)
    else:
        query = attn.to_q(hidden_states)
        key = attn.to_k(hidden_states)
        value = attn.to_v(hidden_states)

    encoder_query = encoder_key = encoder_value = None
    if encoder_hidden_states is not None and getattr(attn, "added_kv_proj_dim", None) is not None:
        if getattr(attn, "fused_projections", False) and hasattr(attn, "to_added_qkv"):
            encoder_query, encoder_key, encoder_value = attn.to_added_qkv(encoder_hidden_states).chunk(3, dim=-1)
        else:
            encoder_query = attn.add_q_proj(encoder_hidden_states)
            encoder_key = attn.add_k_proj(encoder_hidden_states)
            encoder_value = attn.add_v_proj(encoder_hidden_states)

    return query, key, value, encoder_query, encoder_key, encoder_value


class FluxAttentionCaptureProcessor:
    """
    Attention processor for selected FLUX blocks.

    It mirrors the Diffusers FLUX attention processor but replaces fused/SDPA attention
    with explicit q @ k^T softmax so that image-to-image probabilities can be recorded.
    """

    _attention_backend = None

    def __init__(self, store: CaptureStore, block_type: str, block_id: int, text_length: int):
        self.store = store
        self.block_type = block_type
        self.block_id = block_id
        self.text_length = text_length

    def __call__(
        self,
        attn,
        hidden_states: torch.Tensor,
        encoder_hidden_states: Optional[torch.Tensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        image_rotary_emb: Optional[torch.Tensor] = None,
        **kwargs,
    ):
        batch_size = hidden_states.shape[0]
        original_dtype = hidden_states.dtype

        query, key, value, encoder_query, encoder_key, encoder_value = _get_qkv_projections_for_flux(
            attn, hidden_states, encoder_hidden_states
        )

        # Diffusers FLUX layout before attention is [B, sequence, heads, head_dim].
        query = query.unflatten(-1, (attn.heads, -1))
        key = key.unflatten(-1, (attn.heads, -1))
        value = value.unflatten(-1, (attn.heads, -1))

        query = attn.norm_q(query)
        key = attn.norm_k(key)

        if encoder_hidden_states is not None and encoder_query is not None:
            encoder_query = encoder_query.unflatten(-1, (attn.heads, -1))
            encoder_key = encoder_key.unflatten(-1, (attn.heads, -1))
            encoder_value = encoder_value.unflatten(-1, (attn.heads, -1))

            encoder_query = attn.norm_added_q(encoder_query)
            encoder_key = attn.norm_added_k(encoder_key)

            # Joint sequence order in Diffusers FLUX dual blocks is [text, image].
            text_len = encoder_hidden_states.shape[1]
            query = torch.cat([encoder_query, query], dim=1)
            key = torch.cat([encoder_key, key], dim=1)
            value = torch.cat([encoder_value, value], dim=1)
            img_start = text_len
        else:
            # Single blocks receive hidden_states = [text, image].
            img_start = self.text_length if hidden_states.shape[1] > self.text_length else 0

        if image_rotary_emb is not None:
            query = apply_rotary_emb(query, image_rotary_emb, sequence_dim=1)
            key = apply_rotary_emb(key, image_rotary_emb, sequence_dim=1)

        # Convert to [B, H, S, D].
        q = query.transpose(1, 2)
        k = key.transpose(1, 2)
        v = value.transpose(1, 2)

        # Explicit attention. Use fp32 for numerical stability, return output in original dtype.
        scores = torch.matmul(q.float(), k.float().transpose(-1, -2)) * getattr(attn, "scale", q.shape[-1] ** -0.5)
        if attention_mask is not None:
            scores = scores + attention_mask
        probs = torch.softmax(scores, dim=-1)
        out = torch.matmul(probs.to(v.dtype), v)
        out = out.transpose(1, 2).flatten(2, 3).to(original_dtype)  # [B, S, C]

        # Always save the image-image attention probabilities and QKV-derived tensors before output projection.
        if self.store.want_timestep():
            rec = self.store.rec(self.block_type, self.block_id)
            rec.attn_img2img = detach_cpu(probs[:, :, img_start:, img_start:])
            if img_start > 0:
                # Mean over image queries of raw mass flowing to each text token: [B, H, K_text].
                rec.attn_img2text = detach_cpu(probs[:, :, img_start:, :img_start].mean(dim=-2))
            rec.key_img = detach_cpu(k[:, :, img_start:, :])
            rec.value_img = detach_cpu(v[:, :, img_start:, :])

        if encoder_hidden_states is not None:
            # Split [text, image] -> return [image, text], matching Diffusers FluxTransformerBlock.
            encoder_out, image_out = out.split_with_sizes(
                [encoder_hidden_states.shape[1], out.shape[1] - encoder_hidden_states.shape[1]], dim=1
            )
            image_out = attn.to_out[0](image_out)
            image_out = attn.to_out[1](image_out)
            encoder_out = attn.to_add_out(encoder_out)

            # For dual blocks, this is the actual projected attention update added to the image residual.
            if self.store.want_timestep():
                self.store.rec(self.block_type, self.block_id).attention_output = detach_cpu(image_out)

            return image_out, encoder_out
        else:
            # Single blocks have pre_only=True, so no attn.to_out is applied inside attention.
            # The image slice is the attention stream before the block-level proj_out mixes attention and MLP.
            if self.store.want_timestep():
                self.store.rec(self.block_type, self.block_id).attention_output = detach_cpu(out[:, img_start:, :])
            return out


def _first_tensor_from_hook_args(args, kwargs, name="hidden_states"):
    if isinstance(kwargs, dict) and name in kwargs:
        return kwargs[name]
    if args:
        return args[0]
    return None


def run_flux(cfg: ExperimentConfig) -> Dict[Any, BlockCapture]:
    from diffusers import FluxPipeline

    dtype = {"float16": torch.float16, "bfloat16": torch.bfloat16, "float32": torch.float32}[cfg.dtype]
    root = Path(cfg.output_dir)
    root.mkdir(parents=True, exist_ok=True)
    ensure_output_dirs(root)

    pipe = FluxPipeline.from_pretrained(cfg.model_name, torch_dtype=dtype).to(cfg.device)
    pipe.set_progress_bar_config(disable=False)
    transformer = pipe.transformer
    store = CaptureStore(cfg)

    step_state = {"i": -1}

    def to_float(x):
        if x is None:
            return None
        if torch.is_tensor(x):
            return float(x.detach().flatten()[0].cpu())
        try:
            return float(x)
        except Exception:
            return None

    def transformer_pre_hook(module, args, kwargs):
        # This hook fires before each transformer forward call, i.e. before selected blocks run.
        step_state["i"] += 1
        store.current_timestep_index = int(step_state["i"])
        timestep = kwargs.get("timestep", None) if isinstance(kwargs, dict) else None
        store.current_timestep_value = to_float(timestep)
        return None

    store.handles.append(transformer.register_forward_pre_hook(transformer_pre_hook, with_kwargs=True))

    def add_block_hooks(blocks, block_type: str, wanted: Sequence[int]):
        for block_id in wanted:
            block = blocks[block_id]

            def pre_hook(mod, args, kwargs, block_id=block_id, block_type=block_type):
                if not store.want_timestep():
                    return None
                x = _first_tensor_from_hook_args(args, kwargs, "hidden_states")
                if x is not None:
                    store.rec(block_type, block_id).pre_block = detach_cpu(x)
                return None

            def post_hook(mod, args, kwargs, out, block_id=block_id, block_type=block_type):
                if not store.want_timestep():
                    return None
                # FLUX blocks return (encoder_hidden_states, image_hidden_states).
                if isinstance(out, tuple) and len(out) >= 2:
                    image_states = out[1]
                else:
                    image_states = out
                store.rec(block_type, block_id).post_block = detach_cpu(image_states)
                return None

            store.handles.append(block.register_forward_pre_hook(pre_hook, with_kwargs=True))
            store.handles.append(block.register_forward_hook(post_hook, with_kwargs=True))

            # Dual blocks expose the state immediately after attention/residual and before MLP at norm2 input.
            if block_type == "dual" and hasattr(block, "norm2"):
                def pre_norm2_hook(mod, inp, block_id=block_id, block_type=block_type):
                    if store.want_timestep():
                        store.rec(block_type, block_id).post_attention_residual = detach_cpu(inp[0])
                store.handles.append(block.norm2.register_forward_pre_hook(pre_norm2_hook))

    add_block_hooks(transformer.transformer_blocks, "dual", cfg.dual_blocks)
    add_block_hooks(transformer.single_transformer_blocks, "single", cfg.single_blocks)

    # Replace attention processor only for selected blocks.
    for block_id in cfg.dual_blocks:
        transformer.transformer_blocks[block_id].attn.set_processor(
            FluxAttentionCaptureProcessor(store, "dual", block_id, cfg.text_length)
        )
    for block_id in cfg.single_blocks:
        transformer.single_transformer_blocks[block_id].attn.set_processor(
            FluxAttentionCaptureProcessor(store, "single", block_id, cfg.text_length)
        )

    try:
        for prompt_id, prompt in enumerate(cfg.prompts):
            for seed in cfg.seeds:
                store.prompt_id = prompt_id
                store.prompt = prompt
                store.seed = seed
                step_state["i"] = -1
                gen = torch.Generator(device=cfg.device).manual_seed(seed)

                result = pipe(
                    prompt=prompt,
                    height=cfg.height,
                    width=cfg.width,
                    num_inference_steps=cfg.num_inference_steps,
                    guidance_scale=cfg.guidance_scale,
                    generator=gen,
                    max_sequence_length=cfg.text_length,
                    output_type="pil",
                )
                if cfg.save_generated_images:
                    result.images[0].save(root / "generated_images" / f"prompt{prompt_id}_seed{seed}.png")
    finally:
        store.close()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Validate expected captures.
    expected = []
    for prompt_id, prompt in enumerate(cfg.prompts):
        for seed in cfg.seeds:
            for t in cfg.capture_timesteps:
                expected += [(prompt_id, seed, t, "dual", i) for i in cfg.dual_blocks]
                expected += [(prompt_id, seed, t, "single", i) for i in cfg.single_blocks]

    missing = []
    for key in expected:
        rec = store.records.get(key)
        if rec is None:
            missing.append(f"{key}: no record")
            continue
        try:
            validate_capture(rec)
        except Exception as e:
            missing.append(f"{key}: {e}")

    if missing:
        raise RuntimeError("Incomplete capture:\n" + "\n".join(missing[:20]))

    return store.records

In [ ]:
# ================================================================
# Cell 10: Minimal real FLUX run
# ================================================================
# Start with 512x512. After this works, try height=1024,width=1024 if your GPU can handle it.

flux_cfg = ExperimentConfig(
    model_name="black-forest-labs/FLUX.1-schnell",
    prompts=[
        "A close-up portrait of an astronaut in a reflective helmet, cinematic lighting, ultra detailed"
    ],
    seeds=[42],
    num_inference_steps=4,
    capture_timesteps=[3],
    dual_blocks=[18],
    single_blocks=[0],
    guidance_scale=0.0,
    height=512,
    width=512,
    output_dir=PERSISTENT_OUTPUT_DIR,
    device="cuda" if torch.cuda.is_available() else "cpu",
    dtype="bfloat16",
)

root = Path(flux_cfg.output_dir)
root.mkdir(parents=True, exist_ok=True)
ensure_output_dirs(root)
(root / "config.json").write_text(json.dumps(asdict(flux_cfg), indent=2))

records = run_flux(flux_cfg)
head_df, token_df, stage_df = build_metric_tables(flux_cfg, records)
trans_df = transition_metrics(flux_cfg, token_df)

head_df.to_csv(root / "head_level_metrics.csv", index=False)
token_df.to_csv(root / "token_level_metrics.csv", index=False)
stage_df.to_csv(root / "stage_level_metrics.csv", index=False)
trans_df.to_csv(root / "transition_metrics.csv", index=False)
plot_all_outputs(flux_cfg, records, head_df, token_df, stage_df)

print(f"Wrote FLUX outputs to: {root}")
display(head_df.head())

In [ ]:
# ================================================================
# Cell 11: Quick result summary in notebook
# ================================================================
# This cell gives a compact answer to: are high-norm tokens also attention sinks?

root = Path(PERSISTENT_OUTPUT_DIR)
head_df = pd.read_csv(root / "head_level_metrics.csv")
token_df = pd.read_csv(root / "token_level_metrics.csv")
stage_df = pd.read_csv(root / "stage_level_metrics.csv")

rank_cols = [c for c in head_df.columns if c.startswith("sink_rank_by_norm__")]
n_tokens = int(head_df["num_image_tokens"].iloc[0])

summary_rows = []
for c in rank_cols:
    stage = c.replace("sink_rank_by_norm__", "")
    jac = head_df.get(f"top5_jaccard__{stage}", pd.Series(dtype=float))
    summary_rows.append({
        "stage": stage,
        "median_sink_norm_rank": head_df[c].median(),
        "mean_sink_norm_rank": head_df[c].mean(),
        "top5_jaccard": jac.mean(),
        "top5_jaccard_chance": expected_topk_jaccard(min(5, n_tokens), n_tokens),
        "top5_overlap_rate": head_df.get(f"top5_overlap__{stage}", pd.Series(dtype=float)).mean(),
        "mean_spearman_norm_attention": head_df.get(f"spearman_norm_vs_incoming__{stage}", pd.Series(dtype=float)).mean(),
        "frac_heads_spearman_p<0.05": (head_df.get(f"spearman_pvalue__{stage}", pd.Series(dtype=float)) < 0.05).mean(),
    })
summary_df = pd.DataFrame(summary_rows).sort_values("median_sink_norm_rank")
display(summary_df)

if "text_attention_fraction" in head_df.columns:
    print("\nText-attention leak (per block):")
    display(head_df.groupby("block_name").agg(
        mean_text_attention_fraction=("text_attention_fraction", "mean"),
        frac_heads_global_sink_is_text=("global_sink_is_text", "mean"),
    ))
    print("Caution: for heads where global_sink_is_text is True, the image 'sink' from")
    print("renormalized img->img attention is not the head's dominant receiver.")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.barplot(data=summary_df, x="stage", y="top5_jaccard", ax=axes[0])
axes[0].axhline(summary_df["top5_jaccard_chance"].iloc[0], linestyle="--", color="red", label="chance")
axes[0].set_ylim(0, 1)
axes[0].legend()
axes[0].set_title("Top-5 Jaccard: high-norm set vs sink set")
plt.setp(axes[0].get_xticklabels(), rotation=25, ha="right")

sns.barplot(data=summary_df, x="stage", y="mean_spearman_norm_attention", ax=axes[1])
axes[1].axhline(0, linestyle="--", linewidth=1, color="gray")
axes[1].set_title("Mean Spearman: token norm vs incoming attention")
plt.setp(axes[1].get_xticklabels(), rotation=25, ha="right")
fig.tight_layout()
plt.show()


In [ ]:
# ================================================================
# Cell 12: Scale-up experiment template
# ================================================================
# Use this after the minimal run works.
# Example: more prompts, seeds, timesteps, and blocks.

scale_cfg = ExperimentConfig(
    model_name="black-forest-labs/FLUX.1-schnell",
    prompts=[
        "A close-up portrait of an astronaut in a reflective helmet, cinematic lighting, ultra detailed",
        "A futuristic city street at night after rain, neon reflections, detailed architecture",
        "A red fox sitting in a snowy forest, shallow depth of field, cinematic",
    ],
    seeds=[0, 1, 2],
    num_inference_steps=4,
    capture_timesteps=[0, 1, 2, 3],
    dual_blocks=[0, 4, 8, 12, 18],
    single_blocks=[0, 4, 8, 16, 24, 37],
    guidance_scale=0.0,
    height=512,
    width=512,
    output_dir=f"{PERSISTENT_OUTPUT_DIR}_scaleup",
    device="cuda" if torch.cuda.is_available() else "cpu",
    dtype="bfloat16",
)

# Uncomment when ready. This can be slow/heavy.
# root = Path(scale_cfg.output_dir)
# root.mkdir(parents=True, exist_ok=True)
# ensure_output_dirs(root)
# (root / "config.json").write_text(json.dumps(asdict(scale_cfg), indent=2))
# records = run_flux(scale_cfg)
# head_df, token_df, stage_df = build_metric_tables(scale_cfg, records)
# trans_df = transition_metrics(scale_cfg, token_df)
# head_df.to_csv(root / "head_level_metrics.csv", index=False)
# token_df.to_csv(root / "token_level_metrics.csv", index=False)
# stage_df.to_csv(root / "stage_level_metrics.csv", index=False)
# trans_df.to_csv(root / "transition_metrics.csv", index=False)
# plot_all_outputs(scale_cfg, records, head_df, token_df, stage_df)
# print(f"Wrote scale-up outputs to: {root}")

In [ ]:
# ================================================================
# Cell 13: how to interpret the outputs
# ================================================================
# Useful rules of thumb for this experiment:
#
# 1. Strong high-norm/sink coupling:
#    - top5_jaccard clearly above top5_jaccard_chance, especially in
#      attention_output or post_attention_residual.
#    - sink_rank_by_norm close to 1; rank-rank hexbins show a diagonal ridge.
#    - Spearman(norm, incoming attention) positive with p < 0.05 on most heads.
#
# 2. Attention sinks without high-norm coupling:
#    - sink_strength high, but sink_rank_by_norm large and Jaccard at chance.
#    - Spearman near 0 or negative.
#
# 3. High-norm tokens that later become sinks (Dual->Single pathway):
#    - dual18 high post_block norm tokens have low/better incoming_attention_rank in single0/1.
#    - transition_metrics.csv has target_num_heads_top1_sink or target_num_heads_top5_sink > 0.
#    - sink_growth_percentile near 1.0 means the sink token grew more than nearly all
#      non-sink tokens, not just the single matched control.
#
# 4. Text-token confound (check FIRST before interpreting image sinks):
#    - If text_attention_fraction is high (say > 0.5) or global_sink_is_text is True for
#      many heads, the head's true sink lives in the text stream (often T5 padding).
#      Image-token sink claims for those heads describe only the residual image mass.
#
# 5. For FLUX single blocks:
#    - attention and MLP are parallel inside the block, so there is no clean
#      post-attention residual state; use attention_output and post_block separately.
#
# 6. Spatial maps (spatial_maps/):
#    - Overlap category patches at consistent grid locations across prompts/seeds suggest
#      positional (register-like) structure; content-tracking locations suggest semantic sinks.
#    - The *_sink_locations.png map shows whether heads share one global sink or each pick
#      their own; a single bright cell = block-wide shared sink.
#
# 7. This notebook is correlational. Overlap and correlation motivate, but do not
#    establish causal claims. Those need suppression interventions (attention-logit
#    attenuation on sinks, magnitude reduction on high-norm tokens), which build on
#    top of this capture code.

print("Open the saved folders under:", Path(PERSISTENT_OUTPUT_DIR).resolve())
print("Key CSVs: head_level_metrics.csv, token_level_metrics.csv, stage_level_metrics.csv, transition_metrics.csv")
print("Flagship figures: spatial_maps/, overlap_summaries/topk_jaccard_by_stage.png, rank_rank/, text_attention/")


## Cell 14-15: Paper-definition high-norm analysis (added)

Implements a high-norm definition using layer-specific L2-norm percentile thresholds (**97.7th** percentile for middle layers 18 to 39, **99.99th** elsewhere; layer mapping `dual_blockN` → layer N, `single_blockN` → layer 19+N) and applies it to the sink-overlap analysis.

What it adds beyond the earlier top-k metrics:

1. **Option B primary stage**: single blocks thresholded on `attention_output` (where the paper shows high-norms emerge), dual blocks on `post_block` (post-MLP). Tests whether attention builds the high-norm token at the location it drains into (single), and whether attention "marks" a token MLP later amplifies (dual). **Option A** (`post_block` everywhere) is computed alongside as robustness; dual `attention_output` is the paper-predicted negative control.
2. **Percentile sensitivity sweep** (90 / 95 / 97.7 / 99 / 99.9 / 99.99), to check whether the coupling survives a change in the threshold. If Jaccard curves are flat across the sweep, the coupling is not a threshold artifact.
3. **Chi-null substrate test**: under Gaussian residual components, token norms follow σ·χ(D=3072), which predicts a threshold/median ratio of ≈1.026 at the 97.7th percentile. An empirical percentile threshold always selects ~2.3% of tokens *by construction*, so counts alone cannot demonstrate outliers; the test is whether the observed threshold/median ratio far exceeds the chi prediction. Blocks where it matches the null have **no outlier substrate**, the correct reading for late single blocks, rather than "chance-level overlap."
4. **Per-head membership**: the fraction of heads whose top-1 sink token falls inside the paper-percentile high-norm set. This is the number that formalizes the "highest-norm token = sink" impression from the spatial maps.

Point `ANALYSIS_DIR` at the scale-up CSVs on Drive (default) or leave the fallback for local runs. Outputs go to `<ANALYSIS_DIR>/paper_definition_figs/`.


In [ ]:
# ================================================================
# Cell 15: paper-definition high-norm analysis
# ================================================================
# Implements a high-norm definition using layer-specific L2-norm percentile
# thresholds: 97.7th percentile for middle layers (18 to 39 in FLUX global
# indexing), 99.99th elsewhere.
#
# Layer mapping (confirmed): dual_blockN -> layer N; single_blockN -> layer 19+N.
#
# Stage choice (Option B, primary):
#   - single blocks: threshold on `attention_output` norms, where the paper
#     shows high-norms emerge in single blocks. Tests whether attention builds
#     the high-norm token at the same location it drains attention into.
#   - dual blocks: threshold on `post_block` norms (post-MLP), tests the
#     cross-sublayer hypothesis that attention "marks" a token MLP amplifies.
# Option A (post_block for all blocks) is computed alongside as a robustness
# check, and dual `attention_output` is kept as a negative control (the paper
# predicts no high-norm substrate there).
#
# Also produces:
#   - percentile sensitivity sweep (90/95/97.7/99/99.9/99.99), to check whether
#     the coupling survives a change in the threshold
#   - chi-null substrate test: under Gaussian components, norms ~ sigma*chi(D).
#     NOTE an *empirical* percentile threshold always selects ~(100-p)% of
#     tokens by construction, so token *counts* cannot detect outliers.
#     Instead we test whether the threshold-to-median norm RATIO exceeds the
#     chi(D) prediction, and count tokens above the chi-null 99.99% threshold
#     (expected ~0.0001*N under the null).
#
# Self-contained: reads the metric CSVs from ANALYSIS_DIR. Point it at the
# scale-up outputs on Drive, or leave as-is for the minimal-run outputs.

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi as chi_dist

sns.set_theme(context="notebook", style="whitegrid")

# ----------------------------- Config -----------------------------
ANALYSIS_DIR = Path("/content/drive/MyDrive/attnSinks-v-HighNorm_scaleup")
if not ANALYSIS_DIR.exists():
    ANALYSIS_DIR = Path(f"{PERSISTENT_OUTPUT_DIR}_scaleup")
if not ANALYSIS_DIR.exists():
    ANALYSIS_DIR = Path(PERSISTENT_OUTPUT_DIR)

D_MODEL = 3072                      # FLUX hidden dim (norm ~ sigma * chi(D) under Gaussian null)
MIDDLE_LAYERS = (18, 39)            # inclusive global-layer window using 97.7th pct (per the paper)
PCT_MIDDLE, PCT_EDGE = 97.7, 99.99  # paper thresholds
PCT_SWEEP = [90.0, 95.0, 97.7, 99.0, 99.9, 99.99]
STAGES = ["attention_output", "post_block"]

OUT = ANALYSIS_DIR / "paper_definition_figs"
OUT.mkdir(parents=True, exist_ok=True)


def block_to_layer(block_type: str, block_id: int) -> int:
    return int(block_id) if block_type == "dual" else 19 + int(block_id)


def paper_percentile(layer: int) -> float:
    return PCT_MIDDLE if MIDDLE_LAYERS[0] <= layer <= MIDDLE_LAYERS[1] else PCT_EDGE


def primary_stage(block_type: str) -> str:   # Option B
    return "attention_output" if block_type == "single" else "post_block"


def jaccard(a: set, b: set) -> float:
    u = a | b
    return len(a & b) / len(u) if u else float("nan")


def chance_jaccard(k: int, n: int) -> float:
    if k <= 0 or n <= 0:
        return float("nan")
    ei = k * k / n
    return ei / max(2 * k - ei, 1e-9)


# ----------------------------- Load -----------------------------
print(f"Reading from {ANALYSIS_DIR}")
usecols = ["prompt_id", "seed", "timestep_index", "block_type", "block_id",
           "head_id", "token_id", "incoming_attention_mass",
           "norm__attention_output", "norm__post_block"]
token_df = pd.read_csv(ANALYSIS_DIR / "token_level_metrics.csv",
                       usecols=lambda c: c in usecols, low_memory=False)
cap_cols = ["prompt_id", "seed", "timestep_index", "block_type", "block_id"]
print(f"token rows: {len(token_df):,}")

# Per-capture norms (identical across heads -> take one head's rows).
first_head = token_df.groupby(cap_cols)["head_id"].transform("min")
norms_df = token_df[token_df["head_id"] == first_head][
    cap_cols + ["token_id"] + [f"norm__{s}" for s in STAGES if f"norm__{s}" in token_df.columns]
].copy()

# Head-mean incoming attention per token, and each head's top-1 sink token.
incoming_mean = (token_df.groupby(cap_cols + ["token_id"])["incoming_attention_mass"]
                 .mean().rename("incoming_mean").reset_index())
top1_idx = token_df.groupby(cap_cols + ["head_id"])["incoming_attention_mass"].idxmax()
head_top1 = token_df.loc[top1_idx, cap_cols + ["head_id", "token_id"]].rename(
    columns={"token_id": "top1_sink_token"})

# Chi-null reference ratios (threshold/median under Gaussian components).
chi_med = chi_dist.ppf(0.5, D_MODEL)
null_ratio = {p: chi_dist.ppf(p / 100.0, D_MODEL) / chi_med for p in PCT_SWEEP}
null_thr_scale_9999 = chi_dist.ppf(0.9999, D_MODEL) / chi_med  # threshold as multiple of median

# ----------------------------- Per-capture computation -----------------------------
rows, head_rows = [], []
norm_groups = dict(tuple(norms_df.groupby(cap_cols)))
inc_groups = dict(tuple(incoming_mean.groupby(cap_cols)))
h1_groups = dict(tuple(head_top1.groupby(cap_cols)))

for key, gn in norm_groups.items():
    gi = inc_groups.get(key)
    if gi is None:
        continue
    gn = gn.sort_values("token_id")
    gi = gi.sort_values("token_id")
    n = len(gn)
    block_type, block_id = key[3], int(key[4])
    layer = block_to_layer(block_type, block_id)
    p_paper = paper_percentile(layer)
    inc = gi["incoming_mean"].to_numpy()

    for stage in STAGES:
        col = f"norm__{stage}"
        if col not in gn.columns or gn[col].isna().all():
            continue
        norms = gn[col].to_numpy()
        med = float(np.median(norms))

        # chi-null substrate diagnostics (threshold-free of the empirical percentile)
        null_thr = med * null_thr_scale_9999
        n_above_null = int((norms > null_thr).sum())          # expected ~0.0001*n under null

        for p in PCT_SWEEP:
            thr_norm = float(np.percentile(norms, p))
            thr_inc = float(np.percentile(inc, p))
            hn = set(gn["token_id"].to_numpy()[norms > thr_norm].tolist())
            sk = set(gi["token_id"].to_numpy()[inc > thr_inc].tolist())
            rows.append(dict(
                prompt_id=key[0], seed=key[1], timestep_index=key[2],
                block_type=block_type, block_id=block_id,
                block_name=f"{block_type}_block{block_id}", layer=layer, stage=stage,
                percentile=p, is_paper_pct=bool(abs(p - p_paper) < 1e-9),
                is_primary_stage=bool(stage == primary_stage(block_type)),
                n_tokens=n, n_highnorm=len(hn), n_sink=len(sk),
                jaccard=jaccard(hn, sk),
                chance_jaccard=chance_jaccard(max(len(hn), 1), n),
                norm_ratio=thr_norm / med if med > 0 else np.nan,
                chi_null_ratio=null_ratio[p],
                n_above_chi_null=n_above_null,
                chi_null_expected=1e-4 * n,
            ))

        # Per-head top-1 sink membership in the paper-percentile high-norm set.
        gh = h1_groups.get(key)
        if gh is not None:
            thr_paper = float(np.percentile(norms, p_paper))
            hn_paper = set(gn["token_id"].to_numpy()[norms > thr_paper].tolist())
            for _, hr in gh.iterrows():
                head_rows.append(dict(
                    prompt_id=key[0], seed=key[1], timestep_index=key[2],
                    block_name=f"{block_type}_block{block_id}", layer=layer,
                    stage=stage, head_id=int(hr["head_id"]),
                    top1_sink_in_highnorm=bool(int(hr["top1_sink_token"]) in hn_paper),
                ))

sweep_df = pd.DataFrame(rows)
headmem_df = pd.DataFrame(head_rows)
sweep_df.to_csv(OUT / "percentile_sweep.csv", index=False)
headmem_df.to_csv(OUT / "head_top1_in_highnorm.csv", index=False)

# ----------------------------- Printed summaries -----------------------------
print("\n=== 1. Substrate test (chi-null): tokens above the chi(D) 99.99% threshold ===")
print("Under Gaussian residuals, expected count ≈ 0.0001 × n_tokens ≈ 0.1. Large counts = real outlier substrate.")
sub = (sweep_df[sweep_df["percentile"] == PCT_SWEEP[0]]  # one row per capture/stage
       .groupby(["block_name", "layer", "stage"], as_index=False)
       .agg(median_above_null=("n_above_chi_null", "median"),
            max_above_null=("n_above_chi_null", "max")))
print(sub.sort_values(["layer", "stage"]).to_string(index=False))

print("\n=== 2. Paper-definition results (Option B primary stage, paper percentile) ===")
paper_rows = sweep_df[sweep_df["is_paper_pct"] & sweep_df["is_primary_stage"]]
summ = (paper_rows.groupby(["block_name", "layer"], as_index=False)
        .agg(mean_jaccard=("jaccard", "mean"), sd_jaccard=("jaccard", "std"),
             mean_chance=("chance_jaccard", "mean"),
             mean_n_highnorm=("n_highnorm", "mean"),
             mean_norm_ratio=("norm_ratio", "mean"),
             chi_null_ratio=("chi_null_ratio", "mean")))
print(summ.sort_values("layer").to_string(index=False))
print("(norm_ratio >> chi_null_ratio  =>  outliers exceed Gaussian prediction; ≈  =>  no substrate)")

print("\n=== 3. Option A robustness (post_block for all blocks, paper percentile) ===")
optA = sweep_df[sweep_df["is_paper_pct"] & (sweep_df["stage"] == "post_block")]
summA = (optA.groupby(["block_name", "layer"], as_index=False)
         .agg(mean_jaccard=("jaccard", "mean"), mean_norm_ratio=("norm_ratio", "mean")))
print(summA.sort_values("layer").to_string(index=False))

print("\n=== 4. Negative control: dual blocks at attention_output ===")
neg = sweep_df[sweep_df["is_paper_pct"] & (sweep_df["block_type"] == "dual") & (sweep_df["stage"] == "attention_output")]
if not neg.empty:
    print(neg.groupby("block_name")[["jaccard", "norm_ratio", "n_above_chi_null"]].mean().to_string())
    print("(Paper predicts NO high-norm emergence after attention in dual blocks -> expect norm_ratio ≈ chi-null and low counts.)")

if not headmem_df.empty:
    print("\n=== 5. Fraction of heads whose top-1 sink is inside the paper-percentile high-norm set ===")
    hm = (headmem_df[headmem_df.apply(lambda r: r["stage"] == primary_stage("dual" if "dual" in r["block_name"] else "single"), axis=1)]
          .groupby(["block_name", "layer"], as_index=False)["top1_sink_in_highnorm"].mean())
    print(hm.sort_values("layer").to_string(index=False))

# ----------------------------- Figures -----------------------------
# Fig 1: Jaccard vs percentile sensitivity curves.
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8), sharey=True)
for ax, stage in zip(axes, STAGES):
    g = sweep_df[sweep_df["stage"] == stage]
    agg = g.groupby(["block_name", "layer", "percentile"], as_index=False)["jaccard"].mean()
    sns.lineplot(data=agg, x="percentile", y="jaccard", hue="block_name",
                 hue_order=sorted(agg["block_name"].unique(), key=lambda b: block_to_layer("dual" if "dual" in b else "single", int(b.rsplit("block", 1)[-1]))),
                 marker="o", ax=ax)
    ch = g.groupby("percentile", as_index=False)["chance_jaccard"].mean()
    ax.plot(ch["percentile"], ch["chance_jaccard"], "k--", linewidth=1, label="chance")
    for pv in (PCT_MIDDLE, PCT_EDGE):
        ax.axvline(pv, color="gray", linestyle=":", linewidth=1)
    ax.set_title(f"stage = {stage}")
    ax.set_xlabel("high-norm / sink percentile threshold")
    ax.legend(fontsize=7)
axes[0].set_ylabel("Jaccard(high-norm set, sink set)")
fig.suptitle("Sensitivity of norm↔sink overlap to threshold percentile (dotted verticals = paper thresholds)", y=1.03)
fig.tight_layout()
fig.savefig(OUT / "fig1_jaccard_vs_percentile.png", dpi=180, bbox_inches="tight")
plt.show()

# Fig 2: Substrate heatmap, norm-ratio at 97.7th pct vs chi-null, per block x stage.
r977 = sweep_df[np.isclose(sweep_df["percentile"], 97.7)]
piv = r977.pivot_table(index="block_name", columns="stage", values="norm_ratio", aggfunc="mean")
piv = piv.reindex(sorted(piv.index, key=lambda b: block_to_layer("dual" if "dual" in b else "single", int(b.rsplit("block", 1)[-1]))))
plt.figure(figsize=(7.5, max(3, 0.5 * len(piv))))
sns.heatmap(piv, annot=True, fmt=".2f", cmap="magma",
            cbar_kws={"label": "97.7th-pct threshold / median norm"})
plt.title(f"Outlier substrate: threshold/median norm ratio\n(chi({D_MODEL}) null predicts ≈ {null_ratio[97.7]:.3f}; larger = real outliers)")
plt.tight_layout()
plt.savefig(OUT / "fig2_substrate_norm_ratio.png", dpi=180, bbox_inches="tight")
plt.show()

# Fig 3: Fraction of heads whose top-1 sink lies inside the paper high-norm set (primary stage).
if not headmem_df.empty:
    hm_all = headmem_df.copy()
    hm_all["is_primary"] = hm_all.apply(
        lambda r: r["stage"] == primary_stage("dual" if "dual" in r["block_name"] else "single"), axis=1)
    hp = (hm_all[hm_all["is_primary"]]
          .groupby(["block_name", "layer"], as_index=False)["top1_sink_in_highnorm"].mean()
          .sort_values("layer"))
    plt.figure(figsize=(10, 4))
    sns.barplot(data=hp, x="block_name", y="top1_sink_in_highnorm", order=hp["block_name"])
    plt.ylim(0, 1)
    plt.ylabel("fraction of heads")
    plt.xticks(rotation=25, ha="right")
    plt.title("Heads whose top-1 sink token is inside the paper-percentile high-norm set\n(Option B primary stage)")
    plt.tight_layout()
    plt.savefig(OUT / "fig3_head_top1_in_highnorm.png", dpi=180, bbox_inches="tight")
    plt.show()

print(f"\nAll outputs saved to: {OUT}")


## Cells 16-18: Seed-clamp causal intervention

**Hypothesis under test:** dual-MLP-seeded high-norm tokens (layers 17-18) cause single-block attention-sink selection.

**Intervention:** forward pre-hook at the input of `single_transformer_blocks[0]` (= dual→single boundary; `hidden_states` there is the image-only stream). At **every denoising step**, dynamically select tokens with norm > 3× median (cap 8) and rescale them to the median norm, preserving direction.

**Conditions** (run in order; the seed condition logs per-step counts that the controls reuse, so all conditions clamp equal numbers): `clean`, `sham` (select but don't modify, an instrumentation anchor), `seed_clamp`, `matched_clamp` (next-n by norm), `random_clamp` (fixed RNG, excludes seed ranks).

**Outcome table: every result is informative.**

| Seed-clamp observation | Conclusion |
|---|---|
| Heads scatter, agreement drops to late-block levels | Loud tokens necessary for register-sink formation |
| Sinks migrate to the *new* loudest tokens (norm-rank stays ≈1, id changes) | Magnitude-driven selection |
| Sinks stay at the same token ids (rank jumps, id constant) | Position/direction-driven selection |
| text_attention_fraction rises | Image register and T5 padding sink are functional substitutes |
| Image change ≈ matched/random controls | Registers causally confirmed as no-ops for generation |
| Clamped norms regrow by single4/8 | A restoring mechanism exists downstream |

Run cell 17 (uncomment the last line; 3 prompts × 2 seeds × 5 conditions = 30 generations), then cell 18 for the analysis. Requires cells 3-9 to have been run first (uses `ExperimentConfig`, `CaptureStore`, `FluxAttentionCaptureProcessor`, `build_metric_tables`).


In [ ]:
# ================================================================
# Cell 17: seed-clamp intervention, infrastructure and runner
# ================================================================
# Causal test: do dual-MLP-seeded high-norm tokens cause
# single-block sink selection?
#
# Intervention point: forward pre-hook on single_transformer_blocks[k]
# (default k=0). In this Diffusers version the block's `hidden_states` argument
# is the image-only stream, so modifying it = clamping at the dual->single
# boundary, before any single block runs.
#
# Conditions (run in this order; the seed condition logs per-step counts that
# the matched/random controls reuse so all conditions clamp the same number):
#   clean         : no hook modification (baseline)
#   sham          : hook selects tokens but does NOT modify (instrumentation anchor;
#                   must reproduce clean within bf16 noise)
#   seed_clamp    : rescale tokens with norm > seed_ratio × median down to the median norm
#   matched_clamp : same count, next-highest-norm non-seed tokens
#   random_clamp  : same count, uniformly random non-seed tokens (fixed RNG)
#
# Clamping preserves direction: h_i <- h_i * (median_norm / ||h_i||).
# Selection is dynamic per denoising step (the intervention is applied at every
# step, so downstream steps see the intervened trajectory, the correct causal design).

from dataclasses import dataclass, field, asdict
import json as _json

INTERVENTION_CONDITIONS = ["clean", "sham", "seed_clamp", "matched_clamp", "random_clamp"]


@dataclass
class InterventionConfig:
    base: "ExperimentConfig"                  # reuse capture config (blocks to capture, prompts, etc.)
    intervention_single_block: int = 0        # clamp at the input of this single block
    seed_ratio: float = 3.0                   # seed token = norm > seed_ratio × median
    k_max: int = 8                            # cap on tokens clamped per step
    conditions: List[str] = field(default_factory=lambda: list(INTERVENTION_CONDITIONS))
    output_dir: str = ""

    def __post_init__(self):
        if not self.output_dir:
            self.output_dir = f"{self.base.output_dir}_intervention"


class BoundaryClampHook:
    """Forward pre-hook that clamps selected image tokens to the median norm."""

    def __init__(self, icfg: InterventionConfig):
        self.icfg = icfg
        self.condition = "clean"
        self.prompt_id = 0
        self.gen_seed = 0
        self.step = -1                     # set externally per transformer call
        self.seed_counts: Dict[Tuple[int, int, int], int] = {}   # (prompt, seed, step) -> n clamped
        self.log_rows: List[dict] = []

    def _select(self, norms: torch.Tensor) -> Tuple[torch.Tensor, float]:
        """Return (selected token indices, median). norms: [N] float32."""
        med = norms.median()
        key = (self.prompt_id, self.gen_seed, self.step)
        if self.condition == "random_clamp" or self.condition == "matched_clamp":
            n = self.seed_counts.get(key, 0)
        else:
            outlier = torch.nonzero(norms > self.icfg.seed_ratio * med).flatten()
            if outlier.numel() > self.icfg.k_max:
                outlier = outlier[torch.argsort(norms[outlier], descending=True)[: self.icfg.k_max]]
            n = int(outlier.numel())
            if self.condition in ("seed_clamp", "sham"):
                self.seed_counts[key] = n
            if self.condition in ("seed_clamp", "sham", "clean"):
                return outlier, float(med)
        if n == 0:
            return torch.empty(0, dtype=torch.long, device=norms.device), float(med)
        order = torch.argsort(norms, descending=True)
        if self.condition == "matched_clamp":
            sel = order[n: 2 * n]                       # next-n by norm, below the seed set
        else:                                            # random_clamp
            g = torch.Generator(device="cpu").manual_seed(self.gen_seed * 10007 + self.step * 101 + self.prompt_id)
            pool = order[n:].cpu()                       # exclude the seed-rank slots
            sel = pool[torch.randperm(pool.numel(), generator=g)[:n]].to(norms.device)
        return sel, float(med)

    def __call__(self, module, args, kwargs):
        # Locate the image hidden states (kwarg or first positional).
        in_kwargs = isinstance(kwargs, dict) and "hidden_states" in kwargs
        x = kwargs["hidden_states"] if in_kwargs else (args[0] if args else None)
        if x is None:
            return None
        norms = x[0].float().norm(dim=-1)               # [N], B=1 throughout this notebook
        sel, med = self._select(norms)

        self.log_rows.append(dict(
            condition=self.condition, prompt_id=self.prompt_id, seed=self.gen_seed,
            step=self.step, n_selected=int(sel.numel()),
            selected_tokens=_json.dumps([int(t) for t in sel.tolist()]),
            median_norm=med,
            max_norm=float(norms.max()),
            selected_norms=_json.dumps([round(float(norms[t]), 2) for t in sel.tolist()]),
        ))

        if self.condition in ("clean", "sham") or sel.numel() == 0:
            return None                                  # observe only

        scale = (med / norms[sel].clamp(min=1e-6)).to(x.dtype)      # [n]
        x = x.clone()
        x[0, sel, :] = x[0, sel, :] * scale.unsqueeze(-1)
        if in_kwargs:
            kwargs = dict(kwargs); kwargs["hidden_states"] = x
            return (args, kwargs)
        return ((x,) + tuple(args[1:]), kwargs)


def run_intervention_suite(icfg: InterventionConfig) -> None:
    """Run all conditions with one pipeline load. Saves per-condition head/stage CSVs,
    images, a clamp log, and a norm-regrowth trace."""
    from diffusers import FluxPipeline

    cfg = icfg.base
    dtype = {"float16": torch.float16, "bfloat16": torch.bfloat16, "float32": torch.float32}[cfg.dtype]
    root = Path(icfg.output_dir)
    root.mkdir(parents=True, exist_ok=True)
    (root / "config.json").write_text(_json.dumps({**asdict(cfg), "intervention": {
        "single_block": icfg.intervention_single_block, "seed_ratio": icfg.seed_ratio,
        "k_max": icfg.k_max, "conditions": icfg.conditions}}, indent=2))

    pipe = FluxPipeline.from_pretrained(cfg.model_name, torch_dtype=dtype).to(cfg.device)
    pipe.set_progress_bar_config(disable=True)
    transformer = pipe.transformer

    clamp_hook = BoundaryClampHook(icfg)
    regrowth_rows: List[dict] = []

    for condition in icfg.conditions:
        print(f"\n=== Condition: {condition} ===")
        clamp_hook.condition = condition
        store = CaptureStore(cfg)
        step_state = {"i": -1}

        def transformer_pre_hook(module, args, kwargs):
            step_state["i"] += 1
            store.current_timestep_index = int(step_state["i"])
            clamp_hook.step = int(step_state["i"])
            ts = kwargs.get("timestep", None) if isinstance(kwargs, dict) else None
            try:
                store.current_timestep_value = float(ts.detach().flatten()[0].cpu()) if torch.is_tensor(ts) else (float(ts) if ts is not None else None)
            except Exception:
                store.current_timestep_value = None
            return None

        store.handles.append(transformer.register_forward_pre_hook(transformer_pre_hook, with_kwargs=True))
        # Intervention hook FIRST on the target block, so capture hooks see the clamped state.
        store.handles.append(
            transformer.single_transformer_blocks[icfg.intervention_single_block]
            .register_forward_pre_hook(clamp_hook, with_kwargs=True)
        )

        # Reuse the capture machinery from Cell 9.
        def add_block_hooks(blocks, block_type, wanted):
            for block_id in wanted:
                block = blocks[block_id]

                def pre_hook(mod, args, kwargs, block_id=block_id, block_type=block_type):
                    if store.want_timestep():
                        x = _first_tensor_from_hook_args(args, kwargs, "hidden_states")
                        if x is not None:
                            store.rec(block_type, block_id).pre_block = detach_cpu(x)
                    return None

                def post_hook(mod, args, kwargs, out, block_id=block_id, block_type=block_type):
                    if store.want_timestep():
                        image_states = out[1] if isinstance(out, tuple) and len(out) >= 2 else out
                        store.rec(block_type, block_id).post_block = detach_cpu(image_states)
                    return None

                store.handles.append(block.register_forward_pre_hook(pre_hook, with_kwargs=True))
                store.handles.append(block.register_forward_hook(post_hook, with_kwargs=True))
                if block_type == "dual" and hasattr(block, "norm2"):
                    def pre_norm2_hook(mod, inp, block_id=block_id, block_type=block_type):
                        if store.want_timestep():
                            store.rec(block_type, block_id).post_attention_residual = detach_cpu(inp[0])
                    store.handles.append(block.norm2.register_forward_pre_hook(pre_norm2_hook))

        add_block_hooks(transformer.transformer_blocks, "dual", cfg.dual_blocks)
        add_block_hooks(transformer.single_transformer_blocks, "single", cfg.single_blocks)
        for block_id in cfg.dual_blocks:
            transformer.transformer_blocks[block_id].attn.set_processor(
                FluxAttentionCaptureProcessor(store, "dual", block_id, cfg.text_length))
        for block_id in cfg.single_blocks:
            transformer.single_transformer_blocks[block_id].attn.set_processor(
                FluxAttentionCaptureProcessor(store, "single", block_id, cfg.text_length))

        img_dir = root / "images" / condition
        img_dir.mkdir(parents=True, exist_ok=True)
        try:
            for prompt_id, prompt in enumerate(cfg.prompts):
                for gseed in cfg.seeds:
                    store.prompt_id, store.prompt, store.seed = prompt_id, prompt, gseed
                    clamp_hook.prompt_id, clamp_hook.gen_seed = prompt_id, gseed
                    step_state["i"] = -1
                    gen = torch.Generator(device=cfg.device).manual_seed(gseed)
                    result = pipe(prompt=prompt, height=cfg.height, width=cfg.width,
                                  num_inference_steps=cfg.num_inference_steps,
                                  guidance_scale=cfg.guidance_scale, generator=gen,
                                  max_sequence_length=cfg.text_length, output_type="pil")
                    result.images[0].save(img_dir / f"prompt{prompt_id}_seed{gseed}.png")
        finally:
            store.close()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        # Metric tables for this condition (head-level only; token-level is large).
        head_df, token_df, stage_df = build_metric_tables(cfg, store.records)
        head_df.insert(0, "condition", condition)
        head_df.to_csv(root / f"head_metrics_{condition}.csv", index=False)

        # Norm-regrowth trace: norms of THIS condition's clamped tokens at every captured block.
        clog = pd.DataFrame(clamp_hook.log_rows)
        clog_c = clog[clog["condition"] == condition]
        for rec in store.records.values():
            row = clog_c[(clog_c["prompt_id"] == rec.prompt_id) & (clog_c["seed"] == rec.seed)
                         & (clog_c["step"] == rec.timestep_index)]
            if row.empty or rec.post_block is None:
                continue
            ids = _json.loads(row.iloc[0]["selected_tokens"])
            if not ids:
                continue
            norms_post = rec.post_block[0].norm(dim=-1)
            regrowth_rows.append(dict(
                condition=condition, prompt_id=rec.prompt_id, seed=rec.seed,
                timestep_index=rec.timestep_index, block_name=rec.block_name,
                mean_clamped_norm_post=float(norms_post[ids].mean()),
                max_clamped_norm_post=float(norms_post[ids].max()),
                median_norm_post=float(norms_post.median()),
                n_clamped=len(ids),
            ))
        del store, token_df, stage_df

    pd.DataFrame(clamp_hook.log_rows).to_csv(root / "clamp_log.csv", index=False)
    pd.DataFrame(regrowth_rows).to_csv(root / "norm_regrowth.csv", index=False)
    print(f"\nDone. Outputs in {root}")


# ----------------------------- Configure and run -----------------------------
int_base_cfg = ExperimentConfig(
    model_name="black-forest-labs/FLUX.1-schnell",
    prompts=[
        "A close-up portrait of an astronaut in a reflective helmet, cinematic lighting, ultra detailed",
        "A futuristic city street at night after rain, neon reflections, detailed architecture",
        "A red fox sitting in a snowy forest, shallow depth of field, cinematic",
    ],
    seeds=[0, 1],
    num_inference_steps=4,
    capture_timesteps=[0, 1, 2, 3],
    dual_blocks=[18],                       # observe the seeding block too
    single_blocks=[0, 4, 8, 16],            # observe sink behavior downstream of the clamp
    height=512, width=512, guidance_scale=0.0,
    output_dir=f"{PERSISTENT_OUTPUT_DIR}_scaleup",   # base name; suite appends _intervention
    device="cuda" if torch.cuda.is_available() else "cpu", dtype="bfloat16",
)
int_cfg = InterventionConfig(base=int_base_cfg, intervention_single_block=0, seed_ratio=3.0, k_max=8)

# Uncomment to run (3 prompts × 2 seeds × 5 conditions = 30 generations):
#run_intervention_suite(int_cfg)


In [ ]:
# ================================================================
# Cell 18: Intervention analysis
# ================================================================
# Reads the suite outputs and answers, per condition vs clean baseline:
#   0. Sham validation: does observation-only reproduce the baseline?
#   1. Sink fate: vanished / migrated-to-new-loudest / stayed-in-place?
#   2. Head agreement: does the shared-sink structure survive?
#   3. Padding-sink fallback: does attention reroute to the T5 text sink?
#   4. Image effect: LPIPS/MSE vs clean, seed clamp vs matched/random controls
#   5. Norm regrowth: do clamped tokens get re-inflated downstream?

from pathlib import Path
import json as _json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

sns.set_theme(context="notebook", style="whitegrid")

INT_DIR = Path(int_cfg.output_dir) if "int_cfg" in dir() else Path(f"{PERSISTENT_OUTPUT_DIR}_scaleup_intervention")
FIG = INT_DIR / "analysis_figs"
FIG.mkdir(parents=True, exist_ok=True)

conds = [c for c in ["clean", "sham", "seed_clamp", "matched_clamp", "random_clamp"]
         if (INT_DIR / f"head_metrics_{c}.csv").exists()]
head = pd.concat([pd.read_csv(INT_DIR / f"head_metrics_{c}.csv", low_memory=False) for c in conds],
                 ignore_index=True)
clog = pd.read_csv(INT_DIR / "clamp_log.csv")
key_cols = ["prompt_id", "seed", "timestep_index", "block_name", "head_id"]

base = head[head["condition"] == "clean"][key_cols + ["sink_token_id", "sink_strength"]].rename(
    columns={"sink_token_id": "base_sink", "sink_strength": "base_strength"})
head = head.merge(base, on=key_cols, how="left")

# Clamped-token sets per (condition, prompt, seed, step).
clog["token_set"] = clog["selected_tokens"].apply(lambda s: set(_json.loads(s)))
cl_map = {(r.condition, r.prompt_id, r.seed, r.step): r.token_set for r in clog.itertuples()}
def clamped_set(row):
    return cl_map.get((row["condition"], row["prompt_id"], row["seed"], row["timestep_index"]), set())
head["sink_was_clamped_baseline"] = head.apply(
    lambda r: int(r["base_sink"]) in cl_map.get(("seed_clamp", r["prompt_id"], r["seed"], r["timestep_index"]), set()), axis=1)
head["sink_in_own_clamped"] = head.apply(lambda r: int(r["sink_token_id"]) in clamped_set(r), axis=1)
head["sink_same_as_base"] = head["sink_token_id"] == head["base_sink"]

# ---------------- 0. Sham validation ----------------
sham = head[head["condition"] == "sham"]
if not sham.empty:
    agree = sham["sink_same_as_base"].mean()
    print(f"0. SHAM CHECK — fraction of heads with identical sink to clean: {agree:.3f}")
    print("   (Expect ~1.0; bf16 nondeterminism can cause small deviations. If <0.95, investigate before trusting results.)\n")

# ---------------- 1-2. Sink fate + head agreement ----------------
def head_agreement(g):
    n = len(g); u = g["sink_token_id"].nunique()
    return 1 - (u - 1) / max(n - 1, 1)

# Restrict fate analysis to heads whose BASELINE sink was actually clamped in seed_clamp.
fate = head[head["sink_was_clamped_baseline"]].copy()
fate_summary = (fate.groupby(["condition", "block_name"], as_index=False)
                .agg(n_heads=("sink_token_id", "size"),
                     sink_unchanged=("sink_same_as_base", "mean"),
                     new_sink_is_clamped_token=("sink_in_own_clamped", "mean"),
                     new_sink_norm_rank_pre=("sink_rank_by_norm__pre_block", "median"),
                     sink_strength=("sink_strength", "mean"),
                     base_strength=("base_strength", "mean")))
print("1. SINK FATE (heads whose clean-baseline sink was clamped in seed_clamp):")
print(fate_summary.to_string(index=False))
print("""   Reading: sink_unchanged≈1 in seed_clamp -> position-driven selection (clamp didn't matter).
   sink_unchanged≈0 AND new_sink_norm_rank_pre≈1 -> magnitude-driven re-election of the new loudest.
   sink_unchanged≈0 AND high/median rank -> sink structure destroyed.\n""")

agree_df = (head.groupby(["condition", "block_name", "prompt_id", "seed", "timestep_index"])
            .apply(head_agreement).rename("head_agreement").reset_index())
plt.figure(figsize=(12, 4.2))
sns.boxplot(data=agree_df, x="block_name", y="head_agreement", hue="condition", hue_order=conds)
plt.title("Head agreement by condition (does the shared register-sink structure survive the clamp?)")
plt.xticks(rotation=15); plt.tight_layout()
plt.savefig(FIG / "1_head_agreement.png", dpi=180); plt.show()

# ---------------- 3. Padding-sink fallback ----------------
if "text_attention_fraction" in head.columns:
    tf = (head.groupby(["condition", "block_name"], as_index=False)
          .agg(text_frac=("text_attention_fraction", "mean"),
               frac_text_sink=("global_sink_is_text", "mean")))
    print("3. TEXT/PADDING-SINK FALLBACK:")
    print(tf.pivot(index="block_name", columns="condition", values="text_frac").round(3).to_string())
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
    sns.barplot(data=tf, x="block_name", y="text_frac", hue="condition", hue_order=conds, ax=axes[0])
    axes[0].set_title("Image-query attention mass to TEXT tokens")
    sns.barplot(data=tf, x="block_name", y="frac_text_sink", hue="condition", hue_order=conds, ax=axes[1])
    axes[1].set_title("Fraction of heads whose true sink is a text token")
    for ax in axes: ax.tick_params(axis="x", rotation=15)
    fig.tight_layout(); fig.savefig(FIG / "3_text_fallback.png", dpi=180); plt.show()

# ---------------- 4. Image-level effect ----------------
try:
    import lpips
    lpips_net = lpips.LPIPS(net="alex")
    def img_dist(a, b):
        ta = torch.from_numpy(np.array(a)).permute(2, 0, 1).float() / 127.5 - 1
        tb = torch.from_numpy(np.array(b)).permute(2, 0, 1).float() / 127.5 - 1
        with torch.no_grad():
            return float(lpips_net(ta.unsqueeze(0), tb.unsqueeze(0)))
    dist_name = "LPIPS"
except Exception:
    def img_dist(a, b):
        return float(np.mean((np.array(a).astype(np.float32) - np.array(b).astype(np.float32)) ** 2))
    dist_name = "MSE"

img_rows = []
for p in range(len(int_cfg.base.prompts)):
    for s in int_cfg.base.seeds:
        fn = f"prompt{p}_seed{s}.png"
        clean_p = INT_DIR / "images" / "clean" / fn
        if not clean_p.exists():
            continue
        clean_img = Image.open(clean_p)
        for c in conds[1:]:
            fp = INT_DIR / "images" / c / fn
            if fp.exists():
                img_rows.append(dict(condition=c, prompt_id=p, seed=s,
                                     dist=img_dist(clean_img, Image.open(fp))))
img_df = pd.DataFrame(img_rows)
if not img_df.empty:
    print(f"\n4. IMAGE CHANGE vs clean ({dist_name}):")
    print(img_df.groupby("condition")["dist"].agg(["mean", "std", "max"]).to_string())
    plt.figure(figsize=(8, 4))
    sns.barplot(data=img_df, x="condition", y="dist", order=[c for c in conds if c != "clean"])
    plt.ylabel(dist_name); plt.title(f"Image change vs clean baseline ({dist_name})")
    plt.tight_layout(); plt.savefig(FIG / "4_image_change.png", dpi=180); plt.show()

    # Qualitative grid: one row per prompt, columns = conditions (first seed).
    s0 = int_cfg.base.seeds[0]
    nprompts = len(int_cfg.base.prompts)
    fig, axes = plt.subplots(nprompts, len(conds), figsize=(3.1 * len(conds), 3.1 * nprompts), squeeze=False)
    for pi in range(nprompts):
        for ci, c in enumerate(conds):
            ax = axes[pi][ci]
            fp = INT_DIR / "images" / c / f"prompt{pi}_seed{s0}.png"
            if fp.exists():
                ax.imshow(Image.open(fp))
            ax.set_title(c if pi == 0 else "", fontsize=10)
            ax.axis("off")
    fig.suptitle("Qualitative comparison (seed %d)" % s0, y=1.01)
    fig.tight_layout(); fig.savefig(FIG / "4_qualitative_grid.png", dpi=160, bbox_inches="tight"); plt.show()

# ---------------- 5. Norm regrowth ----------------
reg = pd.read_csv(INT_DIR / "norm_regrowth.csv")
if not reg.empty:
    reg["norm_ratio_to_median"] = reg["mean_clamped_norm_post"] / reg["median_norm_post"]
    piv = reg[reg["condition"] == "seed_clamp"].pivot_table(
        index="block_name", columns="timestep_index", values="norm_ratio_to_median", aggfunc="mean")
    print("\n5. NORM REGROWTH (seed_clamp): clamped tokens' post_block norm / median, by block:")
    print(piv.round(2).to_string())
    print("   ≈1 everywhere -> the clamp sticks (no restoring force).")
    print("   Rising with block depth -> something re-inflates the registers downstream (new finding).")
    plt.figure(figsize=(9, 4))
    sns.lineplot(data=reg[reg["condition"].isin(["seed_clamp", "matched_clamp", "random_clamp"])],
                 x="block_name", y="norm_ratio_to_median", hue="condition", marker="o", errorbar=("ci", 95))
    plt.axhline(1.0, ls="--", color="gray")
    plt.title("Do clamped tokens regrow? (post_block norm / median)")
    plt.xticks(rotation=15); plt.tight_layout()
    plt.savefig(FIG / "5_norm_regrowth.png", dpi=180); plt.show()

print(f"\nFigures saved to {FIG}")


In [ ]:
# --- Ensures int_cfg has a resume flag and condition_is_complete is defined,
# so Cell 20 can check condition status. Safe to run more than once. ---

if not hasattr(int_cfg, "resume"):
    int_cfg.resume = True
    print("Patched: int_cfg.resume = True")

if "condition_is_complete" not in globals():
    def condition_is_complete(root, cfg, condition: str) -> bool:
        """A condition counts as done only if its metrics CSV and every image exist."""
        root = Path(root)
        if not (root / f"head_metrics_{condition}.csv").exists():
            return False
        img_dir = root / "images" / condition
        for prompt_id in range(len(cfg.prompts)):
            for gseed in cfg.seeds:
                if not (img_dir / f"prompt{prompt_id}_seed{gseed}.png").exists():
                    return False
        return True
    print("Defined: condition_is_complete()")

# Sanity check: should list your finished conditions as True.
from pathlib import Path
for c in ["clean", "sham", "seed_clamp", "matched_clamp", "random_clamp",
          "seed_rotate", "seed_scramble", "matched_rotate", "random_rotate", "seed_transplant"]:
    print(f"  {c:18s} complete={condition_is_complete(Path(int_cfg.output_dir), int_cfg.base, c)}")

In [ ]:
# ================================================================
# Cell 20: direction ablation, does DIRECTION cause sink selection?
# ================================================================
# Complement to the magnitude clamp (Cell 17). Because AdaLayerNorm and QK-RMSNorm
# erase magnitude before attention, attention can only see a token's DIRECTION.
# Cell 17 killed magnitude and the sink did not move. Here we kill direction.
#
#   condition          direction   magnitude   question
#   ---------------------------------------------------------------------------
#   seed_rotate        random      original    Does direction alone drive sink choice?
#   seed_scramble      random      median      Same, without injecting a 20x random vector.
#                                              Also: does the norm still regrow?
#   matched_rotate     random      original    Control: next-n by norm (non-register).
#   random_rotate      random      original    Control: n random ordinary tokens.
#   seed_transplant    copied      recipient's Is the register direction SUFFICIENT to
#                                              create a sink at a new position?
#                                              (Also separates direction from RoPE position.)
#
# Writes to the SAME directory as Cell 17 (`int_cfg.output_dir`), so all conditions
# sit side by side for analysis. Existing conditions are never touched or rerun.
# Requires: Cells 1-7, 9, and 17 (definitions only - leave Cell 17's run call commented).

import zlib

DIRECTION_CONDITIONS = ["seed_rotate", "seed_scramble", "matched_rotate", "random_rotate", "seed_transplant"]
# Recommended first pass (27 generations at 9 prompt-seed pairs):
#   ["seed_rotate", "seed_scramble", "random_rotate"]


class DirectionInterventionHook:
    """Forward pre-hook that ablates or transplants token DIRECTION at the dual->single boundary."""

    def __init__(self, icfg: "InterventionConfig", ordinary_rank_min: int = 100):
        self.icfg = icfg
        self.condition = "seed_rotate"
        self.prompt_id = 0
        self.gen_seed = 0
        self.step = -1
        self.ordinary_rank_min = ordinary_rank_min      # transplant recipients must be safely ordinary
        self.counts: Dict[Tuple[int, int, int], int] = {}   # (prompt, seed, step) -> n selected by seed_rotate
        self.log_rows: List[dict] = []

    def _rng(self, tag: str) -> torch.Generator:
        """Deterministic across sessions (avoids Python's randomized str hash)."""
        h = zlib.crc32(f"{tag}|{self.condition}|{self.prompt_id}|{self.gen_seed}|{self.step}".encode())
        return torch.Generator(device="cpu").manual_seed(int(h))

    def _select(self, norms: torch.Tensor) -> Tuple[torch.Tensor, float]:
        med = norms.median()
        key = (self.prompt_id, self.gen_seed, self.step)

        if self.condition in ("seed_rotate", "seed_scramble", "seed_transplant"):
            sel = torch.nonzero(norms > self.icfg.seed_ratio * med).flatten()
            if sel.numel() > self.icfg.k_max:
                sel = sel[torch.argsort(norms[sel], descending=True)[: self.icfg.k_max]]
            if self.condition == "seed_rotate":
                self.counts[key] = int(sel.numel())      # controls match seed_rotate's count
            return sel, float(med)

        n = self.counts.get(key, 0)                      # matched_rotate / random_rotate
        if n == 0:
            return torch.empty(0, dtype=torch.long, device=norms.device), float(med)
        order = torch.argsort(norms, descending=True)
        if self.condition == "matched_rotate":
            sel = order[n: 2 * n]
        else:
            pool = order[n:].cpu()
            sel = pool[torch.randperm(pool.numel(), generator=self._rng("randsel"))[:n]].to(norms.device)
        return sel, float(med)

    def __call__(self, module, args, kwargs):
        in_kwargs = isinstance(kwargs, dict) and "hidden_states" in kwargs
        x = kwargs["hidden_states"] if in_kwargs else (args[0] if args else None)
        if x is None:
            return None

        norms = x[0].float().norm(dim=-1)                # [N]
        sel, med = self._select(norms)
        log = dict(condition=self.condition, prompt_id=self.prompt_id, seed=self.gen_seed,
                   step=self.step, n_selected=int(sel.numel()),
                   selected_tokens=_json.dumps([int(t) for t in sel.tolist()]),
                   median_norm=med, max_norm=float(norms.max()),
                   selected_norms=_json.dumps([round(float(norms[t]), 2) for t in sel.tolist()]),
                   donor_token=-1, recipient_token=-1, recipient_norm=float("nan"))

        if sel.numel() == 0:
            self.log_rows.append(log)
            return None

        x = x.clone()
        D = x.shape[-1]

        if self.condition == "seed_transplant":
            # Copy the loudest register token's unit direction onto an ordinary token,
            # scaled to that recipient's OWN norm (no magnitude injected).
            donor = int(sel[torch.argmax(norms[sel])])
            donor_dir = x[0, donor].float() / norms[donor].clamp(min=1e-6)      # [D], unit
            order = torch.argsort(norms, descending=True)
            pool = order[self.ordinary_rank_min:]
            if pool.numel() == 0:
                self.log_rows.append(log)
                return None
            pick = int(torch.randint(pool.numel(), (1,), generator=self._rng("recipient")).item())
            recipient = int(pool[pick])
            recip_norm = norms[recipient]
            x[0, recipient, :] = (donor_dir * recip_norm).to(x.dtype)
            log.update(donor_token=donor, recipient_token=recipient, recipient_norm=float(recip_norm),
                       n_selected=1, selected_tokens=_json.dumps([recipient]))
        else:
            # Replace direction with a random unit vector; choose the target magnitude.
            v = torch.randn(sel.numel(), D, generator=self._rng("dir"))
            v = (v / v.norm(dim=-1, keepdim=True)).to(device=x.device, dtype=torch.float32)
            if self.condition == "seed_scramble":
                target = torch.full((sel.numel(),), med, device=x.device, dtype=torch.float32)
            else:                                        # seed_rotate / matched_rotate / random_rotate
                target = norms[sel].to(torch.float32)
            x[0, sel, :] = (v * target.unsqueeze(-1)).to(x.dtype)

        self.log_rows.append(log)
        if in_kwargs:
            kwargs = dict(kwargs)
            kwargs["hidden_states"] = x
            return (args, kwargs)
        return ((x,) + tuple(args[1:]), kwargs)


def _attach_capture_hooks(transformer, store: CaptureStore, cfg: "ExperimentConfig") -> None:
    """Same capture wiring as Cell 9/17."""
    def add(blocks, block_type, wanted):
        for block_id in wanted:
            block = blocks[block_id]

            def pre_hook(mod, args, kwargs, block_id=block_id, block_type=block_type):
                if store.want_timestep():
                    xx = _first_tensor_from_hook_args(args, kwargs, "hidden_states")
                    if xx is not None:
                        store.rec(block_type, block_id).pre_block = detach_cpu(xx)
                return None

            def post_hook(mod, args, kwargs, out, block_id=block_id, block_type=block_type):
                if store.want_timestep():
                    img = out[1] if isinstance(out, tuple) and len(out) >= 2 else out
                    store.rec(block_type, block_id).post_block = detach_cpu(img)
                return None

            store.handles.append(block.register_forward_pre_hook(pre_hook, with_kwargs=True))
            store.handles.append(block.register_forward_hook(post_hook, with_kwargs=True))
            if block_type == "dual" and hasattr(block, "norm2"):
                def pre_norm2(mod, inp, block_id=block_id, block_type=block_type):
                    if store.want_timestep():
                        store.rec(block_type, block_id).post_attention_residual = detach_cpu(inp[0])
                store.handles.append(block.norm2.register_forward_pre_hook(pre_norm2))

    add(transformer.transformer_blocks, "dual", cfg.dual_blocks)
    add(transformer.single_transformer_blocks, "single", cfg.single_blocks)
    for b in cfg.dual_blocks:
        transformer.transformer_blocks[b].attn.set_processor(
            FluxAttentionCaptureProcessor(store, "dual", b, cfg.text_length))
    for b in cfg.single_blocks:
        transformer.single_transformer_blocks[b].attn.set_processor(
            FluxAttentionCaptureProcessor(store, "single", b, cfg.text_length))


def run_direction_suite(icfg: InterventionConfig, conditions: Optional[List[str]] = None) -> None:
    """Run the direction-ablation conditions. Resume-safe; writes alongside Cell 17's outputs."""
    cfg = icfg.base
    root = Path(icfg.output_dir)
    root.mkdir(parents=True, exist_ok=True)
    conditions = list(conditions) if conditions else list(DIRECTION_CONDITIONS)

    todo, skipped = [], []
    for c in conditions:
        (skipped if (icfg.resume and condition_is_complete(root, cfg, c)) else todo).append(c)
    if skipped:
        print(f"Resume: skipping already-complete conditions {skipped}")
    if not todo:
        print("Nothing to run - all direction conditions already complete.")
        return
    todo.sort(key=lambda c: 0 if c == "seed_rotate" else 1)   # controls need its counts
    print(f"Will run: {todo}  ({len(cfg.prompts) * len(cfg.seeds)} generations each)")

    hook = DirectionInterventionHook(icfg)
    regrowth_rows: List[dict] = []
    dlog_path, dregrow_path = root / "direction_log.csv", root / "direction_regrowth.csv"

    if icfg.resume and dlog_path.exists():
        prior = pd.read_csv(dlog_path)
        for r in prior[prior["condition"] == "seed_rotate"].itertuples():
            hook.counts[(int(r.prompt_id), int(r.seed), int(r.step))] = int(r.n_selected)
        hook.log_rows = prior[prior["condition"].isin(skipped)].to_dict("records")
        print(f"Resume: restored {len(hook.counts)} seed_rotate count entries")
    if icfg.resume and dregrow_path.exists():
        pr = pd.read_csv(dregrow_path)
        regrowth_rows = pr[pr["condition"].isin(skipped)].to_dict("records")

    if ({"matched_rotate", "random_rotate"} & set(todo)) and not hook.counts and "seed_rotate" not in todo:
        raise RuntimeError("Rotate controls need seed_rotate's per-step counts. Run seed_rotate first "
                           "(or keep direction_log.csv from a prior run).")

    from diffusers import FluxPipeline
    dtype = {"float16": torch.float16, "bfloat16": torch.bfloat16, "float32": torch.float32}[cfg.dtype]
    pipe = FluxPipeline.from_pretrained(cfg.model_name, torch_dtype=dtype).to(cfg.device)
    pipe.set_progress_bar_config(disable=True)
    transformer = pipe.transformer

    for condition in todo:
        print(f"\n=== Condition: {condition} ===")
        hook.condition = condition
        store = CaptureStore(cfg)
        step_state = {"i": -1}

        def transformer_pre_hook(module, args, kwargs):
            step_state["i"] += 1
            store.current_timestep_index = int(step_state["i"])
            hook.step = int(step_state["i"])
            ts = kwargs.get("timestep", None) if isinstance(kwargs, dict) else None
            try:
                store.current_timestep_value = float(ts.detach().flatten()[0].cpu()) if torch.is_tensor(ts) else (float(ts) if ts is not None else None)
            except Exception:
                store.current_timestep_value = None
            return None

        store.handles.append(transformer.register_forward_pre_hook(transformer_pre_hook, with_kwargs=True))
        store.handles.append(
            transformer.single_transformer_blocks[icfg.intervention_single_block]
            .register_forward_pre_hook(hook, with_kwargs=True))
        _attach_capture_hooks(transformer, store, cfg)

        img_dir = root / "images" / condition
        img_dir.mkdir(parents=True, exist_ok=True)
        try:
            for prompt_id, prompt in enumerate(cfg.prompts):
                for gseed in cfg.seeds:
                    store.prompt_id, store.prompt, store.seed = prompt_id, prompt, gseed
                    hook.prompt_id, hook.gen_seed = prompt_id, gseed
                    step_state["i"] = -1
                    gen = torch.Generator(device=cfg.device).manual_seed(gseed)
                    res = pipe(prompt=prompt, height=cfg.height, width=cfg.width,
                               num_inference_steps=cfg.num_inference_steps,
                               guidance_scale=cfg.guidance_scale, generator=gen,
                               max_sequence_length=cfg.text_length, output_type="pil")
                    res.images[0].save(img_dir / f"prompt{prompt_id}_seed{gseed}.png")
        finally:
            store.close()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        head_df, token_df, stage_df = build_metric_tables(cfg, store.records)
        head_df.insert(0, "condition", condition)
        head_df.to_csv(root / f"head_metrics_{condition}.csv", index=False)

        # Regrowth of the perturbed tokens.
        dlog = pd.DataFrame(hook.log_rows)
        dcur = dlog[dlog["condition"] == condition]
        for rec in store.records.values():
            row = dcur[(dcur["prompt_id"] == rec.prompt_id) & (dcur["seed"] == rec.seed)
                       & (dcur["step"] == rec.timestep_index)]
            if row.empty or rec.post_block is None:
                continue
            ids = _json.loads(row.iloc[0]["selected_tokens"])
            if not ids:
                continue
            npost = rec.post_block[0].norm(dim=-1)
            regrowth_rows.append(dict(
                condition=condition, prompt_id=rec.prompt_id, seed=rec.seed,
                timestep_index=rec.timestep_index, block_name=rec.block_name,
                mean_perturbed_norm_post=float(npost[ids].mean()),
                median_norm_post=float(npost.median()), n_perturbed=len(ids)))
        del store, token_df, stage_df

    pd.DataFrame(hook.log_rows).drop_duplicates(
        subset=["condition", "prompt_id", "seed", "step"], keep="last").to_csv(dlog_path, index=False)
    if regrowth_rows:
        pd.DataFrame(regrowth_rows).drop_duplicates(
            subset=["condition", "prompt_id", "seed", "timestep_index", "block_name"],
            keep="last").to_csv(dregrow_path, index=False)
    print(f"\nDone. Ran {todo}; skipped {skipped}. Outputs in {root}")


# ----------------------------- Run -----------------------------
# IMPORTANT: reuses `int_cfg` from Cell 17 unchanged, so outputs land next to the
# existing clean/sham/seed_clamp/matched_clamp/random_clamp results and stay comparable.
# Do NOT edit int_cfg (prompts/seeds/blocks) or the old conditions become incomparable.

# Recommended first pass (3 conditions):
#run_direction_suite(int_cfg, conditions=["seed_rotate", "seed_scramble", "random_rotate"])

# Full set (adds matched control + the sufficiency test):
#run_direction_suite(int_cfg)

In [ ]:
# ================================================================
# Cell 21: magnitude vs direction, comparative analysis
# ================================================================
# Loads every condition present on disk (Cell 17's + Cell 20's) and answers:
#   1. Sink fate: does killing MAGNITUDE move the sink? does killing DIRECTION?
#   2. Regrowth: is the register encoded in direction? (clamp regrows, scramble shouldn't)
#   3. Transplant: is the register direction SUFFICIENT to create a sink elsewhere?
#      (and does the sink follow direction, or stay with RoPE position?)
#   4. Text/padding fallback and image damage across all conditions.
# Reads only, no generation.

from pathlib import Path
import json as _j
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from PIL import Image
from scipy.stats import wilcoxon

sns.set_theme(context="notebook", style="whitegrid")

ROOT = Path(int_cfg.output_dir)
FIG = ROOT / "analysis_figs"; FIG.mkdir(parents=True, exist_ok=True)

ALL_CONDS = ["clean", "sham", "seed_clamp", "matched_clamp", "random_clamp",
             "seed_rotate", "seed_scramble", "matched_rotate", "random_rotate", "seed_transplant"]
conds = [c for c in ALL_CONDS if (ROOT / f"head_metrics_{c}.csv").exists()]
print(f"Conditions found: {conds}\n")

head = pd.concat([pd.read_csv(ROOT / f"head_metrics_{c}.csv", low_memory=False) for c in conds],
                 ignore_index=True)

# Merge both perturbation logs.
logs = []
for f, cols in [("clamp_log.csv", None), ("direction_log.csv", None)]:
    p = ROOT / f
    if p.exists():
        logs.append(pd.read_csv(p))
plog = pd.concat(logs, ignore_index=True) if logs else pd.DataFrame()
plog["token_set"] = plog["selected_tokens"].apply(lambda s: set(_j.loads(s)))
pmap = {(r.condition, r.prompt_id, r.seed, r.step): r.token_set for r in plog.itertuples()}

BASE = "sham" if "sham" in conds else "clean"     # sham is bit-identical to clean
key_cols = ["prompt_id", "seed", "timestep_index", "block_name", "head_id"]
base_df = head[head["condition"] == BASE][key_cols + ["sink_token_id"]].rename(columns={"sink_token_id": "base_sink"})
head = head.merge(base_df, on=key_cols, how="left")
head["sink_same_as_base"] = head["sink_token_id"] == head["base_sink"]

# Which baseline sinks were register tokens? (use seed_clamp's selection as the register set)
REG_REF = "seed_clamp" if "seed_clamp" in conds else "seed_rotate"
head["base_sink_is_register"] = head.apply(
    lambda r: int(r["base_sink"]) in pmap.get((REG_REF, r["prompt_id"], r["seed"], r["timestep_index"]), set())
    if pd.notna(r["base_sink"]) else False, axis=1)

blocks = sorted(head["block_name"].unique(),
                key=lambda b: (0 if "dual" in b else 1, int(b.rsplit("block", 1)[-1])))

# ---------------- 1. Sink fate: magnitude vs direction ----------------
fate = head[head["base_sink_is_register"] & (head["condition"] != BASE)]
tbl = (fate.groupby(["condition", "block_name"], as_index=False)
       .agg(n_heads=("sink_token_id", "size"),
            sink_unchanged=("sink_same_as_base", "mean"),
            sink_strength=("sink_strength", "mean")))
piv = tbl.pivot(index="condition", columns="block_name", values="sink_unchanged").reindex(columns=blocks)
print("1. SINK FATE — fraction of heads still sinking at the SAME token as baseline")
print("   (rows = what we destroyed; heads whose baseline sink was a register token)")
print(piv.round(3).to_string())
print("""
   seed_clamp   (magnitude killed, direction kept)  -> HIGH means magnitude is causally inert
   seed_rotate  (direction killed, magnitude kept)  -> LOW  means direction drives sink choice
   seed_scramble(both killed)                       -> should match seed_rotate if direction is the cause
   *_rotate/_clamp controls                         -> should stay HIGH (we never touched their sinks)
""")

plt.figure(figsize=(12, 4.5))
order = [c for c in ["seed_clamp", "seed_rotate", "seed_scramble", "matched_clamp",
                     "matched_rotate", "random_clamp", "random_rotate"] if c in tbl["condition"].unique()]
sns.barplot(data=tbl[tbl["condition"].isin(order)], x="block_name", y="sink_unchanged",
            hue="condition", hue_order=order, order=blocks)
plt.ylim(0, 1.05); plt.ylabel("sink unchanged vs baseline")
plt.title("Does the sink move? Magnitude ablation vs direction ablation")
plt.xticks(rotation=15); plt.tight_layout()
plt.savefig(FIG / "6_sink_fate_magnitude_vs_direction.png", dpi=180); plt.show()

# ---------------- 2. Regrowth: is the register carried by direction? ----------------
reg_frames = []
if (ROOT / "norm_regrowth.csv").exists():
    r1 = pd.read_csv(ROOT / "norm_regrowth.csv").rename(columns={"mean_clamped_norm_post": "mean_perturbed_norm_post",
                                                                 "n_clamped": "n_perturbed"})
    reg_frames.append(r1)
if (ROOT / "direction_regrowth.csv").exists():
    reg_frames.append(pd.read_csv(ROOT / "direction_regrowth.csv"))
if reg_frames:
    reg = pd.concat(reg_frames, ignore_index=True)
    reg["ratio"] = reg["mean_perturbed_norm_post"] / reg["median_norm_post"]
    reg = reg[reg["block_name"] != "dual_block18"]     # dual18 runs BEFORE the hook: pre-intervention state
    rp = reg.pivot_table(index="condition", columns="block_name", values="ratio", aggfunc="mean")
    rp = rp.reindex(columns=[b for b in blocks if b in rp.columns])
    print("\n2. NORM REGROWTH — perturbed tokens' post_block norm / median")
    print("   (dual_block18 excluded: it runs before the intervention point)")
    print(rp.round(2).to_string())
    print("""
   seed_clamp    regrows (direction intact -> MLPs refill along it)
   seed_scramble stays ~1 -> the register is encoded in DIRECTION, not magnitude
   seed_scramble regrows  -> something re-creates the direction downstream
""")
    plt.figure(figsize=(10, 4.2))
    sns.lineplot(data=reg[reg["condition"].isin(["seed_clamp", "seed_rotate", "seed_scramble", "random_clamp", "random_rotate"])],
                 x="block_name", y="ratio", hue="condition", marker="o", errorbar=("ci", 95))
    plt.axhline(1.0, ls="--", color="gray")
    plt.title("Do perturbed tokens regrow? (direction intact vs destroyed)")
    plt.xticks(rotation=15); plt.tight_layout()
    plt.savefig(FIG / "7_regrowth_direction_vs_magnitude.png", dpi=180); plt.show()

# ---------------- 3. Transplant: is direction sufficient? ----------------
if "seed_transplant" in conds:
    dl = pd.read_csv(ROOT / "direction_log.csv")
    tl = dl[dl["condition"] == "seed_transplant"]
    rec_map = {(r.prompt_id, r.seed, r.step): (int(r.recipient_token), int(r.donor_token))
               for r in tl.itertuples() if int(r.recipient_token) >= 0}

    tp = head[head["condition"] == "seed_transplant"].copy()
    tp["recipient"] = tp.apply(lambda r: rec_map.get((r["prompt_id"], r["seed"], r["timestep_index"]), (-1, -1))[0], axis=1)
    tp["donor"] = tp.apply(lambda r: rec_map.get((r["prompt_id"], r["seed"], r["timestep_index"]), (-1, -1))[1], axis=1)
    tp = tp[tp["recipient"] >= 0]
    tp["sink_is_recipient"] = tp["sink_token_id"] == tp["recipient"]
    tp["sink_is_donor"] = tp["sink_token_id"] == tp["donor"]

    # Baseline rate: how often is that same recipient token the sink WITHOUT the transplant?
    bl = head[head["condition"] == BASE].merge(
        tp[key_cols + ["recipient", "donor"]], on=key_cols, how="inner")
    bl["sink_is_recipient"] = bl["sink_token_id"] == bl["recipient"]

    out = pd.DataFrame({
        "transplant_sink_is_recipient": tp.groupby("block_name")["sink_is_recipient"].mean(),
        "baseline_sink_is_recipient": bl.groupby("block_name")["sink_is_recipient"].mean(),
        "transplant_sink_is_donor": tp.groupby("block_name")["sink_is_donor"].mean(),
    }).reindex(blocks).dropna(how="all")
    print("\n3. TRANSPLANT — register direction copied onto an ordinary token (at its own norm)")
    print(out.round(4).to_string())
    print("""
   recipient rate >> baseline   -> direction is SUFFICIENT to create a sink (sink follows direction)
   recipient rate ~ baseline, donor still sink -> selection is positional (RoPE), not directional
""")
    plt.figure(figsize=(9, 4))
    out[["transplant_sink_is_recipient", "baseline_sink_is_recipient"]].plot(kind="bar", ax=plt.gca())
    plt.ylabel("fraction of heads"); plt.title("Does the sink follow a transplanted direction?")
    plt.xticks(rotation=15); plt.tight_layout()
    plt.savefig(FIG / "8_transplant_sufficiency.png", dpi=180); plt.show()

# ---------------- 4. Text fallback ----------------
if "text_attention_fraction" in head.columns:
    tf = head.groupby(["condition", "block_name"], as_index=False)["text_attention_fraction"].mean()
    tp2 = tf.pivot(index="block_name", columns="condition", values="text_attention_fraction").reindex(blocks)
    print("\n4. TEXT/PADDING FALLBACK — image-query attention mass to text tokens")
    print(tp2.round(3).to_string())
    plt.figure(figsize=(12, 4.2))
    sns.barplot(data=tf, x="block_name", y="text_attention_fraction", hue="condition",
                hue_order=conds, order=blocks)
    plt.title("Attention rerouted to the T5 padding sink, by condition")
    plt.xticks(rotation=15); plt.tight_layout()
    plt.savefig(FIG / "9_text_fallback_all.png", dpi=180); plt.show()

# ---------------- 5. Image damage + paired test ----------------
try:
    import lpips
    _net = lpips.LPIPS(net="alex")
    def dist(a, b):
        ta = torch.from_numpy(np.array(a)).permute(2, 0, 1).float() / 127.5 - 1
        tb = torch.from_numpy(np.array(b)).permute(2, 0, 1).float() / 127.5 - 1
        with torch.no_grad():
            return float(_net(ta.unsqueeze(0), tb.unsqueeze(0)))
    DNAME = "LPIPS"
except Exception:
    def dist(a, b):
        return float(np.mean((np.array(a).astype(np.float32) - np.array(b).astype(np.float32)) ** 2))
    DNAME = "MSE"
    print("\n(lpips not installed — using MSE. `pip install lpips` for the perceptual metric.)")

rows = []
for p in range(len(int_cfg.base.prompts)):
    for s in int_cfg.base.seeds:
        bp = ROOT / "images" / BASE / f"prompt{p}_seed{s}.png"
        if not bp.exists():
            continue
        bimg = Image.open(bp)
        for c in conds:
            if c == BASE:
                continue
            fp = ROOT / "images" / c / f"prompt{p}_seed{s}.png"
            if fp.exists():
                rows.append(dict(condition=c, pair=f"p{p}_s{s}", dist=dist(bimg, Image.open(fp))))
idf = pd.DataFrame(rows)
if not idf.empty:
    print(f"\n5. IMAGE CHANGE vs {BASE} ({DNAME}):")
    print(idf.groupby("condition")["dist"].agg(["mean", "std", "count"]).round(3).to_string())
    # Paired tests: each register ablation against its matched control.
    for a, b in [("seed_clamp", "matched_clamp"), ("seed_rotate", "matched_rotate"),
                 ("seed_clamp", "random_clamp"), ("seed_rotate", "random_rotate"),
                 ("seed_rotate", "seed_clamp")]:
        if a in conds and b in conds:
            m = idf[idf["condition"] == a].set_index("pair")["dist"].align(
                idf[idf["condition"] == b].set_index("pair")["dist"], join="inner")
            if len(m[0]) >= 5:
                try:
                    st, pv = wilcoxon(m[0].values, m[1].values)
                    print(f"   {a} vs {b}: n={len(m[0])}, median diff={np.median(m[0].values - m[1].values):.3f}, Wilcoxon p={pv:.4f}")
                except Exception as e:
                    print(f"   {a} vs {b}: test failed ({e})")
            else:
                print(f"   {a} vs {b}: n={len(m[0])} pairs — too few for a paired test")
    plt.figure(figsize=(10, 4))
    sns.barplot(data=idf, x="condition", y="dist", order=[c for c in conds if c != BASE])
    plt.ylabel(DNAME); plt.title(f"Image change vs {BASE} ({DNAME})")
    plt.xticks(rotation=20); plt.tight_layout()
    plt.savefig(FIG / "10_image_change_all.png", dpi=180); plt.show()

print(f"\nFigures saved to {FIG}")

In [ ]:
# ================================================================
# Cell 22: qualitative comparison, magnitude vs direction
# ================================================================
# Two figures per seed:
#   A. Image grid   : rows = prompts, columns = conditions, in 2x2-logic order.
#   B. Difference maps: |condition - baseline|, autoscaled so the ~5% changes
#      are actually visible, with the distance metric in each title.
# Read-only. Requires `int_cfg` (Cell 17). Run after Cell 20 has produced images.

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

ROOT = Path(int_cfg.output_dir)
FIG = ROOT / "analysis_figs"; FIG.mkdir(parents=True, exist_ok=True)

# Ordered so the logic reads left-to-right:
#   baseline | magnitude killed | direction killed | both | sufficiency | controls
COND_ORDER = ["clean", "sham",
              "seed_clamp",                       # direction kept, magnitude killed
              "seed_rotate",                      # direction killed, magnitude kept
              "seed_scramble",                    # both killed
              "seed_transplant",                  # direction copied elsewhere
              "matched_clamp", "matched_rotate", "random_clamp", "random_rotate"]

COND_SUBTITLE = {
    "clean": "baseline", "sham": "baseline (hook, no edit)",
    "seed_clamp": "mag→median, dir kept", "seed_rotate": "dir→random, mag kept",
    "seed_scramble": "dir→random, mag→median", "seed_transplant": "dir→ordinary token",
    "matched_clamp": "control", "matched_rotate": "control",
    "random_clamp": "control", "random_rotate": "control",
}

# Distance metric (LPIPS if available, else MSE).
try:
    import lpips, torch as _t
    _net = lpips.LPIPS(net="alex", verbose=False)
    def _dist(a, b):
        ta = _t.from_numpy(np.array(a)).permute(2, 0, 1).float() / 127.5 - 1
        tb = _t.from_numpy(np.array(b)).permute(2, 0, 1).float() / 127.5 - 1
        with _t.no_grad():
            return float(_net(ta.unsqueeze(0), tb.unsqueeze(0)))
    _DN = "LPIPS"
except Exception:
    def _dist(a, b):
        return float(np.mean((np.array(a).astype(np.float32) - np.array(b).astype(np.float32)) ** 2))
    _DN = "MSE"


def qualitative_figures(seed, conditions=None, prompts=None, save=True):
    """Grid + difference maps for one seed."""
    conds = [c for c in (conditions or COND_ORDER)
             if (ROOT / "images" / c).exists()
             and any((ROOT / "images" / c).glob(f"*_seed{seed}.png"))]
    if not conds:
        print(f"No images found for seed {seed}.")
        return
    base = "sham" if "sham" in conds else ("clean" if "clean" in conds else conds[0])
    prompt_ids = prompts if prompts is not None else list(range(len(int_cfg.base.prompts)))

    def load(c, p):
        fp = ROOT / "images" / c / f"prompt{p}_seed{seed}.png"
        return Image.open(fp).convert("RGB") if fp.exists() else None

    # ---------- A. Image grid ----------
    nrow, ncol = len(prompt_ids), len(conds)
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.0 * ncol, 3.2 * nrow), squeeze=False)
    for ri, p in enumerate(prompt_ids):
        bimg = load(base, p)
        for ci, c in enumerate(conds):
            ax = axes[ri][ci]
            img = load(c, p)
            if img is not None:
                ax.imshow(img)
                if ri == 0:
                    ax.set_title(f"{c}\n{COND_SUBTITLE.get(c, '')}", fontsize=9)
                if bimg is not None and c != base:
                    ax.set_xlabel(f"{_DN} {_dist(bimg, img):.3f}", fontsize=8)
            ax.set_xticks([]); ax.set_yticks([])
            for s in ax.spines.values():
                s.set_visible(False)
        axes[ri][0].set_ylabel(f"prompt {p}", fontsize=9)
    fig.suptitle(f"Magnitude vs direction ablation — seed {seed}  (baseline = {base})", y=1.005, fontsize=13)
    fig.tight_layout()
    if save:
        fig.savefig(FIG / f"11_qualitative_grid_seed{seed}.png", dpi=160, bbox_inches="tight")
    plt.show()

    # ---------- B. Difference maps ----------
    diff_conds = [c for c in conds if c != base]
    if not diff_conds:
        return
    fig, axes = plt.subplots(len(prompt_ids), len(diff_conds),
                             figsize=(3.0 * len(diff_conds), 3.2 * len(prompt_ids)), squeeze=False)
    for ri, p in enumerate(prompt_ids):
        bimg = load(base, p)
        if bimg is None:
            continue
        barr = np.array(bimg).astype(np.float32)
        # Common scale per row so conditions are comparable within a prompt.
        diffs = {}
        for c in diff_conds:
            img = load(c, p)
            if img is not None:
                diffs[c] = np.abs(np.array(img).astype(np.float32) - barr).mean(axis=2)
        vmax = max((np.percentile(d, 99.5) for d in diffs.values()), default=1.0) or 1.0
        for ci, c in enumerate(diff_conds):
            ax = axes[ri][ci]
            if c in diffs:
                im = ax.imshow(diffs[c], cmap="inferno", vmin=0, vmax=vmax)
                if ri == 0:
                    ax.set_title(f"{c}\n{COND_SUBTITLE.get(c, '')}", fontsize=9)
                ax.set_xlabel(f"mean |Δ| = {diffs[c].mean():.2f}", fontsize=8)
            ax.set_xticks([]); ax.set_yticks([])
        axes[ri][0].set_ylabel(f"prompt {p}", fontsize=9)
        fig.colorbar(im, ax=axes[ri].tolist(), fraction=0.015, pad=0.01)
    fig.suptitle(f"Where does the image change? |condition − {base}|, seed {seed}\n"
                 f"(each row shares a colour scale; bright = changed)", y=1.005, fontsize=13)
    if save:
        fig.savefig(FIG / f"12_difference_maps_seed{seed}.png", dpi=160, bbox_inches="tight")
    plt.show()


# ----------------------------- Run -----------------------------
for _s in int_cfg.base.seeds:
    qualitative_figures(_s)

# Narrower comparisons, e.g. just the 2x2 logic:
# qualitative_figures(0, conditions=["sham", "seed_clamp", "seed_rotate", "seed_scramble"])

print(f"\nSaved to {FIG}")

In [ ]:
# ================================================================
# Cell 23: extract v* and test the channel bridge (Run A)
# ================================================================
# One clean pass (no interventions). At a set of focus blocks, capture:
#   - register-token full hidden vectors  (register = norm > 3x median, cap 8)
#   - per-channel token-mean-abs profile  [C]         (landscape measurement)
#   - spike positions of candidate channels           (the channel-bridge test)
# Then compute:
#   1. v*: is it ONE direction? (cosine across prompts/seeds/timesteps/blocks)
#   2. Sparsity: which channels carry v*? top-k energy, participation ratio
#   3. Channel bridge: are v*'s top channels among the known candidate set {154, 1446, 1831, 2566}?
#   4. Spike-position test: are the top-1% |x[:,c]| tokens exactly the registers?
# Requires: Cells 1-7, 9 (definitions). ~9 generations. Resume-safe per generation.

FOCUS_BLOCKS = [("dual", 16), ("dual", 17), ("dual", 18),
                ("single", 0), ("single", 4), ("single", 8), ("single", 16)]
CANDIDATE_CHANNELS = [154, 1446, 1831, 2566]      # known global set (Flux-Dev); test transfer
SEED_RATIO_V, KMAX_V = 3.0, 8

VSTAR_DIR = Path(f"{PERSISTENT_OUTPUT_DIR}_scaleup_intervention") / "vstar"
VSTAR_DIR.mkdir(parents=True, exist_ok=True)


def run_vstar_capture(cfg: "ExperimentConfig") -> None:
    from diffusers import FluxPipeline
    dtype = {"float16": torch.float16, "bfloat16": torch.bfloat16, "float32": torch.float32}[cfg.dtype]
    pipe = FluxPipeline.from_pretrained(cfg.model_name, torch_dtype=dtype).to(cfg.device)
    pipe.set_progress_bar_config(disable=True)
    tr = pipe.transformer

    state = {"step": -1, "out": {}}   # out[(btype,bid,step)] = dict of compact stats

    def make_post(btype, bid):
        def post(mod, args, kwargs, out):
            img = out[1] if isinstance(out, tuple) and len(out) >= 2 else out
            h = img[0].float()                                   # [N, C]
            norms = h.norm(dim=-1)
            med = norms.median()
            reg = torch.nonzero(norms > SEED_RATIO_V * med).flatten()
            if reg.numel() > KMAX_V:
                reg = reg[torch.argsort(norms[reg], descending=True)[:KMAX_V]]
            ch_mean_abs = h.abs().mean(dim=0)                    # [C]
            k1 = max(1, h.shape[0] // 100)                       # top-1% positions per candidate channel
            spikes = {c: torch.topk(h[:, c].abs(), k1).indices.cpu() for c in CANDIDATE_CHANNELS}
            state["out"][(btype, bid, state["step"])] = dict(
                register_ids=reg.cpu(), register_vecs=h[reg].cpu(),
                median_norm=float(med), max_norm=float(norms.max()),
                ch_mean_abs=ch_mean_abs.cpu(), spikes=spikes,
            )
        return post

    handles = [tr.register_forward_pre_hook(
        lambda m, a, k: state.__setitem__("step", state["step"] + 1), with_kwargs=True)]
    for btype, bid in FOCUS_BLOCKS:
        blocks = tr.transformer_blocks if btype == "dual" else tr.single_transformer_blocks
        handles.append(blocks[bid].register_forward_hook(make_post(btype, bid), with_kwargs=True))

    try:
        for pid, prompt in enumerate(cfg.prompts):
            for gseed in cfg.seeds:
                fp = VSTAR_DIR / f"capA_p{pid}_s{gseed}.pt"
                if fp.exists():
                    print(f"skip p{pid}s{gseed} (exists)")
                    continue
                state["step"], state["out"] = -1, {}
                gen = torch.Generator(device=cfg.device).manual_seed(gseed)
                pipe(prompt=prompt, height=cfg.height, width=cfg.width,
                     num_inference_steps=cfg.num_inference_steps, guidance_scale=cfg.guidance_scale,
                     generator=gen, max_sequence_length=cfg.text_length, output_type="pil")
                torch.save({"prompt_id": pid, "seed": gseed, "data": state["out"]}, fp)
                print(f"saved {fp.name}")
    finally:
        for h in handles:
            h.remove()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def analyze_vstar() -> None:
    files = sorted(VSTAR_DIR.glob("capA_*.pt"))
    if not files:
        print("No captures found — run run_vstar_capture first.")
        return
    caps = [torch.load(f, weights_only=False) for f in files]

    # ---- 1. Is v* one direction? ----
    unit_vecs, meta = [], []
    for cap in caps:
        for (bt, bid, st), d in cap["data"].items():
            for i in range(d["register_vecs"].shape[0]):
                v = d["register_vecs"][i]
                unit_vecs.append(v / v.norm().clamp(min=1e-6))
                meta.append((cap["prompt_id"], cap["seed"], bt, bid, st))
    if not unit_vecs:
        print("No register tokens found at ratio 3x — lower SEED_RATIO_V?")
        return
    U = torch.stack(unit_vecs)                                     # [M, C]
    meta_df = pd.DataFrame(meta, columns=["prompt_id", "seed", "btype", "bid", "step"])
    sign = torch.sign(U @ U[0])                                    # align sign before averaging
    Ua = U * sign.unsqueeze(-1)
    vstar = Ua.mean(dim=0); vstar = vstar / vstar.norm()
    cos_to_vstar = Ua @ vstar
    print(f"Register unit vectors: M={len(U)}")
    print(f"1. GLOBAL v*: cos(register, v*) mean={cos_to_vstar.mean():.3f}, "
          f"p05={cos_to_vstar.quantile(0.05):.3f}, min={cos_to_vstar.min():.3f}")
    meta_df["cos_vstar"] = cos_to_vstar.numpy()
    print(meta_df.groupby(["btype", "bid"])["cos_vstar"].agg(["mean", "min", "count"]).round(3).to_string())
    print("   (mean cos > ~0.9 everywhere -> ONE shared direction; block-dependent -> a family)")

    # ---- 2. Sparsity of v* ----
    e = vstar.pow(2)
    order = torch.argsort(e, descending=True)
    top = order[:20].tolist()
    cume = torch.cumsum(e[order], 0)
    pr = float(1.0 / e.pow(2).sum())                               # participation ratio of energy
    print(f"\n2. SPARSITY: top-1 ch energy={e[order[0]]:.3f}, top-5 cum={cume[4]:.3f}, "
          f"top-10 cum={cume[9]:.3f}, top-50 cum={cume[49]:.3f}, participation ratio={pr:.1f}")
    print(f"   v* top-20 channels: {top}")

    # ---- 3. Channel bridge to the candidate set ----
    hits = [c for c in CANDIDATE_CHANNELS if c in order[:10].tolist()]
    print(f"\n3. CHANNEL BRIDGE: of candidate channels {CANDIDATE_CHANNELS}, in v* top-10: {hits}")
    for c in CANDIDATE_CHANNELS:
        rank = int((order == c).nonzero().flatten()[0]) + 1
        print(f"   channel {c}: v*-energy rank {rank}, energy fraction {e[c]:.4f}")
    # our own global channel profile (all tokens), for the landscape comparison
    ch_prof = torch.stack([d["ch_mean_abs"] for cap in caps for d in cap["data"].values()]).mean(0)
    top_global = torch.argsort(ch_prof, descending=True)[:10].tolist()
    print(f"   Our token-mean-abs top-10 channels (schnell): {top_global}")

    # ---- 4. Spike-position test (Finding-4 bridge) ----
    rows = []
    for cap in caps:
        for (bt, bid, st), d in cap["data"].items():
            reg = set(d["register_ids"].tolist())
            if not reg:
                continue
            for c, sp in d["spikes"].items():
                s = set(sp.tolist())
                inter = len(reg & s)
                rows.append(dict(btype=bt, bid=bid, channel=c,
                                 frac_registers_in_spikes=inter / len(reg),
                                 jaccard=inter / len(reg | s)))
    sp_df = pd.DataFrame(rows)
    print("\n4. SPIKE-POSITION TEST: are register tokens the spike positions of these channels?")
    print(sp_df.groupby(["channel"])[["frac_registers_in_spikes", "jaccard"]].mean().round(3).to_string())
    print("   frac≈1 for a channel -> registers ARE that channel's spikes (token/channel identity).")

    torch.save({"vstar": vstar, "top_channels": top, "meta": meta_df}, VSTAR_DIR / "vstar.pt")
    print(f"\nSaved v* to {VSTAR_DIR/'vstar.pt'}")

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].hist(cos_to_vstar.numpy(), bins=40)
    axes[0].set_title("cos(register vector, v*)"); axes[0].set_xlabel("cosine")
    axes[1].semilogy(e[order][:200].numpy(), marker=".")
    for c in CANDIDATE_CHANNELS:
        r = int((order == c).nonzero().flatten()[0])
        if r < 200:
            axes[1].annotate(str(c), (r, float(e[c])), fontsize=8, color="red")
    axes[1].set_title("v* energy by channel rank (top 200)"); axes[1].set_xlabel("rank")
    axes[2].semilogy(ch_prof.numpy(), lw=0.4)
    for c in CANDIDATE_CHANNELS:
        axes[2].annotate(str(c), (c, float(ch_prof[c])), fontsize=8, color="red")
    axes[2].set_title("token-mean |x| per channel (our landscape)")
    fig.tight_layout(); fig.savefig(VSTAR_DIR / "fig_vstar_identity.png", dpi=180); plt.show()


# ---- run ----
#run_vstar_capture(int_cfg.base)
#analyze_vstar()

In [ ]:
# ================================================================
# Cell 24: where is v* born, and who writes it? (Run B)
# ================================================================
# Uses the fixed v* from Cell 23. One clean pass capturing, at EVERY block
# (19 dual + 38 single), each token's [norm, projection onto v*] , tiny memory.
# At dual 15-18 additionally captures the post-attention residual, giving the
# exact decomposition per block:  Δattn = postAttn − pre,  Δmlp = post − postAttn,
# each projected onto v*.
# Answers:
#   birth layer: where do eventual-register tokens first separate in v*-component?
#   selection: does the pre-birth v*-component predict WHICH tokens become registers?
#   amplifier law: is a single block's delta(v*-component) proportional to existing component?
# Requires Cell 23's vstar.pt. ~9 generations. Resume-safe per generation.

VS = torch.load(VSTAR_DIR / "vstar.pt", weights_only=False)
VSTAR = VS["vstar"]                                              # [C], unit
RUNB_DIR = VSTAR_DIR / "runB"; RUNB_DIR.mkdir(exist_ok=True)
ATTRIB_DUALS = [15, 16, 17, 18]


def run_birth_capture(cfg: "ExperimentConfig") -> None:
    from diffusers import FluxPipeline
    dtype = {"float16": torch.float16, "bfloat16": torch.bfloat16, "float32": torch.float32}[cfg.dtype]
    pipe = FluxPipeline.from_pretrained(cfg.model_name, torch_dtype=dtype).to(cfg.device)
    pipe.set_progress_bar_config(disable=True)
    tr = pipe.transformer
    vs = VSTAR.to(cfg.device)

    state = {"step": -1, "post": {}, "postattn": {}}

    def stats(h):                                                # h: [N, C] on GPU
        hf = h.float()
        return torch.stack([hf.norm(dim=-1), hf @ vs], dim=1).cpu()   # [N, 2]

    def make_post(layer, btype, bid):
        def post(mod, args, kwargs, out):
            img = out[1] if isinstance(out, tuple) and len(out) >= 2 else out
            state["post"][(layer, state["step"])] = stats(img[0])
        return post

    def make_norm2pre(layer):
        def pre(mod, inp):
            state["postattn"][(layer, state["step"])] = stats(inp[0][0])
        return pre

    handles = [tr.register_forward_pre_hook(
        lambda m, a, k: state.__setitem__("step", state["step"] + 1), with_kwargs=True)]
    for bid, blk in enumerate(tr.transformer_blocks):            # layers 0-18
        handles.append(blk.register_forward_hook(make_post(bid, "dual", bid), with_kwargs=True))
        if bid in ATTRIB_DUALS and hasattr(blk, "norm2"):
            handles.append(blk.norm2.register_forward_pre_hook(make_norm2pre(bid)))
    for bid, blk in enumerate(tr.single_transformer_blocks):     # layers 19-56
        handles.append(blk.register_forward_hook(make_post(19 + bid, "single", bid), with_kwargs=True))

    try:
        for pid, prompt in enumerate(cfg.prompts):
            for gseed in cfg.seeds:
                fp = RUNB_DIR / f"capB_p{pid}_s{gseed}.pt"
                if fp.exists():
                    print(f"skip p{pid}s{gseed}")
                    continue
                state["step"], state["post"], state["postattn"] = -1, {}, {}
                gen = torch.Generator(device=cfg.device).manual_seed(gseed)
                pipe(prompt=prompt, height=cfg.height, width=cfg.width,
                     num_inference_steps=cfg.num_inference_steps, guidance_scale=cfg.guidance_scale,
                     generator=gen, max_sequence_length=cfg.text_length, output_type="pil")
                torch.save({"prompt_id": pid, "seed": gseed,
                            "post": state["post"], "postattn": state["postattn"]}, fp)
                print(f"saved {fp.name}")
    finally:
        for h in handles:
            h.remove()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def analyze_birth(register_layer: int = 19, ratio: float = 3.0) -> None:
    files = sorted(RUNB_DIR.glob("capB_*.pt"))
    caps = [torch.load(f, weights_only=False) for f in files]
    n_layers = 57
    OUT = VSTAR_DIR

    # ---- birth trajectory ----
    traj_reg, traj_ord, birth_rows, sel_rows = [], [], [], []
    for cap in caps:
        steps = sorted({k[1] for k in cap["post"].keys()})
        for st in steps:
            A = torch.stack([cap["post"][(L, st)] for L in range(n_layers)])   # [57, N, 2]
            norms, proj = A[..., 0], A[..., 1].abs()
            nr = norms[register_layer]
            reg = torch.nonzero(nr > ratio * nr.median()).flatten()
            if reg.numel() == 0:
                continue
            ordinary = torch.tensor([t for t in range(norms.shape[1]) if t not in set(reg.tolist())])
            med_prof = proj[:, ordinary].median(dim=1).values                  # [57]
            traj_reg.append(proj[:, reg].mean(dim=1)); traj_ord.append(med_prof)
            # birth = first layer where every register token exceeds 5x ordinary median (persisting)
            thresh = 5.0 * med_prof.clamp(min=1e-6)
            above = (proj[:, reg] > thresh.unsqueeze(1)).all(dim=1)
            born = int(above.nonzero().flatten()[0]) if above.any() else -1
            birth_rows.append(dict(prompt_id=cap["prompt_id"], seed=cap["seed"], step=st, birth_layer=born,
                                   n_registers=int(reg.numel())))
            # selection: does the PRE-birth component predict which tokens become registers?
            if born > 0:
                pre = proj[max(born - 2, 0)]                                   # 2 layers before birth
                ranks = pre.argsort(descending=True)
                reg_ranks = torch.tensor([int((ranks == t).nonzero()) for t in reg.tolist()])
                sel_rows.append(dict(step=st, born=born,
                                     median_prebirth_rank_of_registers=float(reg_ranks.float().median()),
                                     n=int(reg.numel())))
    birth_df, sel_df = pd.DataFrame(birth_rows), pd.DataFrame(sel_rows)
    print("Q1a BIRTH LAYER (first layer where all eventual registers exceed 5x ordinary v*-component):")
    print(birth_df["birth_layer"].value_counts().sort_index().to_string())
    print("\nQ1a' SELECTION — 2 layers pre-birth, registers' rank by v*-component "
          "(low = predestined, ~N/2 = random):")
    if not sel_df.empty:
        print(sel_df[["median_prebirth_rank_of_registers", "n"]].describe().loc[["mean", "50%"]].round(1).to_string())

    R, O = torch.stack(traj_reg).mean(0), torch.stack(traj_ord).mean(0)
    plt.figure(figsize=(11, 4))
    plt.semilogy(R.numpy(), label="eventual register tokens (mean |proj v*|)")
    plt.semilogy(O.numpy(), label="ordinary tokens (median)")
    plt.axvline(18.5, ls="--", color="gray"); plt.text(18.6, R.max() * 0.5, "dual→single", fontsize=8)
    plt.xlabel("layer (0-18 dual, 19-56 single)"); plt.ylabel("|projection onto v*|")
    plt.legend(); plt.title("Birth and life of the register direction")
    plt.tight_layout(); plt.savefig(OUT / "fig_birth_trajectory.png", dpi=180); plt.show()

    # ---- sublayer attribution + amplifier law at dual 15-18 ----
    amp_rows = []
    for cap in caps:
        steps = sorted({k[1] for k in cap["post"].keys()})
        for st in steps:
            for L in ATTRIB_DUALS:
                if (L, st) not in cap["postattn"]:
                    continue
                pre = cap["post"][(L - 1, st)] if L > 0 else None
                pa, po = cap["postattn"][(L, st)], cap["post"][(L, st)]
                if pre is None:
                    continue
                d_attn, d_mlp = pa[:, 1] - pre[:, 1], po[:, 1] - pa[:, 1]
                x = pre[:, 1]
                for name, d in [("attn", d_attn), ("mlp", d_mlp)]:
                    # amplifier law: slope of Δproj vs existing proj (robust, top-decile weighted)
                    xs, ds = x.abs(), d * torch.sign(x + 1e-9)
                    k = max(8, int(0.01 * len(xs)))
                    top = torch.topk(xs, k).indices
                    slope_top = float((ds[top] / xs[top].clamp(min=1e-6)).median())
                    amp_rows.append(dict(layer=L, sublayer=name, step=st,
                                         mean_abs_delta_top1pct=float(ds[top].abs().mean()),
                                         gain_on_existing_component=slope_top))
    amp_df = pd.DataFrame(amp_rows)
    print("\nQ1b AMPLIFIER LAW at dual 15-18 (top-1% v*-component tokens):")
    piv = amp_df.pivot_table(index=["layer", "sublayer"],
                             values=["mean_abs_delta_top1pct", "gain_on_existing_component"], aggfunc="mean")
    print(piv.round(3).to_string())
    print("   gain > 0 for MLP and ≈ 0 for attn at 17-18 -> MLP is a directional amplifier, "
          "attention writes ~nothing along v* (matches magnitude-blindness).")
    amp_df.to_csv(OUT / "amplifier_law.csv", index=False)
    birth_df.to_csv(OUT / "birth_layers.csv", index=False)


# ---- run ----
#run_birth_capture(int_cfg.base)
#analyze_birth()

In [ ]:
# ================================================================
# Cell 25: inertia dose-response, does magnitude protect direction?
# ================================================================
# Transplant v* (the DONOR's live unit direction) onto an ordinary token at
# magnitude alpha × median, for alpha in {1, 3, 6, 12}. One extra condition
# `transplant_iso12` scrambles the donor simultaneously, removing the
# donor-vs-recipient competition so persistence is measured in isolation.
# Predictions (inertia hypothesis):
#   - block-0 capture rate ~ alpha-independent (LayerNorm erases magnitude)
#   - persistence at blocks 4/8/16 increases monotonically with alpha
#   - iso12 >> plain a12 at block 0 (competition removed), same persistence slope
# Reuses: int_cfg, condition_is_complete, CaptureStore, capture processors,
# build_metric_tables, _attach_capture_hooks (Cell 20), _first_tensor_from_hook_args.

DOSE_ALPHAS = {"transplant_a1": 1.0, "transplant_a3": 3.0, "transplant_a6": 6.0,
               "transplant_a12": 12.0, "transplant_iso12": 12.0}


class DoseTransplantHook:
    def __init__(self, icfg: "InterventionConfig", ordinary_rank_min: int = 100):
        self.icfg = icfg; self.ordinary_rank_min = ordinary_rank_min
        self.condition = "transplant_a1"; self.prompt_id = 0; self.gen_seed = 0; self.step = -1
        self.log_rows: List[dict] = []

    def _rng(self, tag):
        h = zlib.crc32(f"{tag}|{self.prompt_id}|{self.gen_seed}|{self.step}".encode())
        return torch.Generator(device="cpu").manual_seed(int(h))       # same recipient across alphas!

    def __call__(self, module, args, kwargs):
        in_kwargs = isinstance(kwargs, dict) and "hidden_states" in kwargs
        x = kwargs["hidden_states"] if in_kwargs else (args[0] if args else None)
        if x is None:
            return None
        norms = x[0].float().norm(dim=-1)
        med = norms.median()
        reg = torch.nonzero(norms > self.icfg.seed_ratio * med).flatten()
        if reg.numel() == 0:
            self.log_rows.append(dict(condition=self.condition, prompt_id=self.prompt_id,
                                      seed=self.gen_seed, step=self.step, n_selected=0,
                                      selected_tokens="[]", donor_token=-1, recipient_token=-1,
                                      median_norm=float(med)))
            return None
        donor = int(reg[torch.argmax(norms[reg])])
        donor_dir = x[0, donor].float() / norms[donor].clamp(min=1e-6)
        order = torch.argsort(norms, descending=True)
        pool = order[self.ordinary_rank_min:]
        recipient = int(pool[int(torch.randint(pool.numel(), (1,), generator=self._rng("recip")).item())])
        alpha = DOSE_ALPHAS[self.condition]

        x = x.clone()
        x[0, recipient, :] = (donor_dir * (alpha * med)).to(x.dtype)
        if self.condition == "transplant_iso12":                      # remove the competitor
            v = torch.randn(x.shape[-1], generator=self._rng("iso"))
            v = v / v.norm()
            x[0, donor, :] = (v.to(x.device) * med).to(x.dtype)

        self.log_rows.append(dict(condition=self.condition, prompt_id=self.prompt_id,
                                  seed=self.gen_seed, step=self.step, n_selected=1,
                                  selected_tokens=_json.dumps([recipient]),
                                  donor_token=donor, recipient_token=recipient,
                                  median_norm=float(med)))
        if in_kwargs:
            kwargs = dict(kwargs); kwargs["hidden_states"] = x
            return (args, kwargs)
        return ((x,) + tuple(args[1:]), kwargs)


def run_dose_suite(icfg: "InterventionConfig", conditions=None) -> None:
    cfg = icfg.base
    root = Path(icfg.output_dir); root.mkdir(parents=True, exist_ok=True)
    conditions = list(conditions) if conditions else list(DOSE_ALPHAS.keys())
    todo = [c for c in conditions if not (getattr(icfg, "resume", True) and condition_is_complete(root, cfg, c))]
    skipped = [c for c in conditions if c not in todo]
    if skipped:
        print(f"Resume: skipping {skipped}")
    if not todo:
        print("All dose conditions complete."); return
    print(f"Will run: {todo}")

    hook = DoseTransplantHook(icfg)
    dlog_path = root / "dose_log.csv"
    if getattr(icfg, "resume", True) and dlog_path.exists():
        hook.log_rows = pd.read_csv(dlog_path).query("condition in @skipped").to_dict("records")

    from diffusers import FluxPipeline
    dtype = {"float16": torch.float16, "bfloat16": torch.bfloat16, "float32": torch.float32}[cfg.dtype]
    pipe = FluxPipeline.from_pretrained(cfg.model_name, torch_dtype=dtype).to(cfg.device)
    pipe.set_progress_bar_config(disable=True)
    tr = pipe.transformer

    for condition in todo:
        print(f"\n=== {condition} (alpha={DOSE_ALPHAS[condition]}) ===")
        hook.condition = condition
        store = CaptureStore(cfg)
        step_state = {"i": -1}

        def tf_pre(m, a, k):
            step_state["i"] += 1
            store.current_timestep_index = step_state["i"]; hook.step = step_state["i"]
            store.current_timestep_value = None
            return None

        store.handles.append(tr.register_forward_pre_hook(tf_pre, with_kwargs=True))
        store.handles.append(tr.single_transformer_blocks[icfg.intervention_single_block]
                             .register_forward_pre_hook(hook, with_kwargs=True))
        _attach_capture_hooks(tr, store, cfg)

        img_dir = root / "images" / condition; img_dir.mkdir(parents=True, exist_ok=True)
        try:
            for pid, prompt in enumerate(cfg.prompts):
                for gseed in cfg.seeds:
                    store.prompt_id, store.prompt, store.seed = pid, prompt, gseed
                    hook.prompt_id, hook.gen_seed = pid, gseed
                    step_state["i"] = -1
                    gen = torch.Generator(device=cfg.device).manual_seed(gseed)
                    res = pipe(prompt=prompt, height=cfg.height, width=cfg.width,
                               num_inference_steps=cfg.num_inference_steps,
                               guidance_scale=cfg.guidance_scale, generator=gen,
                               max_sequence_length=cfg.text_length, output_type="pil")
                    res.images[0].save(img_dir / f"prompt{pid}_seed{gseed}.png")
        finally:
            store.close()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        head_df, _, _ = build_metric_tables(cfg, store.records)
        head_df.insert(0, "condition", condition)
        head_df.to_csv(root / f"head_metrics_{condition}.csv", index=False)
        del store

    pd.DataFrame(hook.log_rows).drop_duplicates(
        subset=["condition", "prompt_id", "seed", "step"], keep="last").to_csv(dlog_path, index=False)
    print("Done.")


def analyze_dose(icfg: "InterventionConfig") -> None:
    root = Path(icfg.output_dir)
    dl = pd.read_csv(root / "dose_log.csv")
    rows = []
    for cond in DOSE_ALPHAS:
        f = root / f"head_metrics_{cond}.csv"
        if not f.exists():
            continue
        hd = pd.read_csv(f, low_memory=False)
        lm = dl[dl["condition"] == cond]
        rmap = {(r.prompt_id, r.seed, r.step): (int(r.recipient_token), int(r.donor_token))
                for r in lm.itertuples() if int(r.recipient_token) >= 0}
        hd["recip"] = hd.apply(lambda r: rmap.get((r["prompt_id"], r["seed"], r["timestep_index"]), (-1, -1))[0], axis=1)
        hd["donor"] = hd.apply(lambda r: rmap.get((r["prompt_id"], r["seed"], r["timestep_index"]), (-1, -1))[1], axis=1)
        hd = hd[hd["recip"] >= 0]
        g = hd.groupby("block_name").apply(
            lambda x: pd.Series({"recipient_rate": (x["sink_token_id"] == x["recip"]).mean(),
                                 "donor_rate": (x["sink_token_id"] == x["donor"]).mean()}))
        for bn, r in g.iterrows():
            rows.append(dict(condition=cond, alpha=DOSE_ALPHAS[cond], block_name=bn, **r.to_dict()))
    df = pd.DataFrame(rows)
    print("DOSE-RESPONSE: fraction of heads sinking at the transplanted recipient")
    print(df.pivot_table(index=["condition", "alpha"], columns="block_name",
                         values="recipient_rate").round(3).to_string())
    df.to_csv(root / "dose_response.csv", index=False)

    blocks = [b for b in ["single_block0", "single_block4", "single_block8", "single_block16"]
              if b in df["block_name"].unique()]
    plt.figure(figsize=(9, 4.5))
    for bn in blocks:
        sub = df[(df["block_name"] == bn) & (df["condition"] != "transplant_iso12")].sort_values("alpha")
        plt.plot(sub["alpha"], sub["recipient_rate"], marker="o", label=bn)
    iso = df[df["condition"] == "transplant_iso12"]
    for bn in blocks:
        s = iso[iso["block_name"] == bn]
        if not s.empty:
            plt.scatter([12], s["recipient_rate"], marker="*", s=160, zorder=5)
    plt.xlabel("transplant magnitude (× median norm)"); plt.ylabel("recipient sink capture")
    plt.title("Inertia test: capture at block 0 should be flat; persistence should rise with α\n(★ = donor removed)")
    plt.legend(fontsize=8); plt.tight_layout()
    plt.savefig(root / "analysis_figs" / "13_dose_response.png", dpi=180); plt.show()


# ---- run ----
#run_dose_suite(int_cfg)
#analyze_dose(int_cfg)



In [ ]:
# ================================================================
# Cell 26: why does attention like v*? Query-key geometry
# ================================================================
# One clean generation; a side-capture processor stores per head:
#   mean image-query vector [H, Dh]  and  all image keys [H, N, Dh]
# (both AFTER QK-RMSNorm, i.e., exactly what enters the logits).
# Then per head: cos(mean_query, key_t) for all t, where does the register rank?

QK_BLOCKS = [("single", 0), ("single", 4), ("single", 8)]
QK_DIR = Path(int_cfg.output_dir) / "qk_geometry"; QK_DIR.mkdir(parents=True, exist_ok=True)


class QKCaptureProcessor:
    """Minimal processor: computes attention normally via SDPA, stores normalized q-mean and keys."""
    def __init__(self, sink, tag, text_length):
        self.sink, self.tag, self.text_length = sink, tag, text_length

    def __call__(self, attn, hidden_states, encoder_hidden_states=None, attention_mask=None,
                 image_rotary_emb=None, **kwargs):
        from diffusers.models.embeddings import apply_rotary_emb
        import torch.nn.functional as F
        x = hidden_states
        ctx = encoder_hidden_states
        if ctx is not None:                                           # dual-style call
            q = attn.to_q(x); k = attn.to_k(x); v = attn.to_v(x)
        else:
            q = attn.to_q(x); k = attn.to_k(x); v = attn.to_v(x)
        B, S, _ = q.shape
        H = attn.heads; Dh = q.shape[-1] // H
        q = q.view(B, S, H, Dh); k = k.view(B, S, H, Dh); v = v.view(B, S, H, Dh)
        if getattr(attn, "norm_q", None) is not None:
            q = attn.norm_q(q)
        if getattr(attn, "norm_k", None) is not None:
            k = attn.norm_k(k)
        if image_rotary_emb is not None:
            q = apply_rotary_emb(q, image_rotary_emb, sequence_dim=1)
            k = apply_rotary_emb(k, image_rotary_emb, sequence_dim=1)
        img_start = self.text_length if S > self.text_length else 0
        self.sink[self.tag] = dict(
            q_mean=q[0, img_start:].mean(dim=0).float().cpu(),        # [H, Dh]
            keys=k[0, img_start:].permute(1, 0, 2).float().cpu(),     # [H, N, Dh]
        )
        q = q.permute(0, 2, 1, 3); k = k.permute(0, 2, 1, 3); v = v.permute(0, 2, 1, 3)
        out = F.scaled_dot_product_attention(q, k, v)
        out = out.permute(0, 2, 1, 3).reshape(B, S, H * Dh).to(hidden_states.dtype)
        return out


def run_qk_capture(cfg: "ExperimentConfig", prompt_id=0, gseed=0):
    from diffusers import FluxPipeline
    dtype = {"float16": torch.float16, "bfloat16": torch.bfloat16, "float32": torch.float32}[cfg.dtype]
    pipe = FluxPipeline.from_pretrained(cfg.model_name, torch_dtype=dtype).to(cfg.device)
    pipe.set_progress_bar_config(disable=True)
    tr = pipe.transformer
    sink, regs = {}, {}
    orig = {}
    for bt, bid in QK_BLOCKS:
        blk = tr.single_transformer_blocks[bid]
        orig[(bt, bid)] = blk.attn.processor
        blk.attn.set_processor(QKCaptureProcessor(sink, f"{bt}{bid}", cfg.text_length))

        def make_pre(tag):
            def pre(mod, args, kwargs):
                x = kwargs.get("hidden_states", args[0] if args else None)
                n = x[0].float().norm(dim=-1)
                r = torch.nonzero(n > 3.0 * n.median()).flatten()
                regs[tag] = r[torch.argsort(n[r], descending=True)][:8].cpu()
                return None
            return pre
        blk.register_forward_pre_hook(make_pre(f"{bt}{bid}"), with_kwargs=True)

    gen = torch.Generator(device=cfg.device).manual_seed(gseed)
    pipe(prompt=cfg.prompts[prompt_id], height=cfg.height, width=cfg.width,
         num_inference_steps=cfg.num_inference_steps, guidance_scale=cfg.guidance_scale,
         generator=gen, max_sequence_length=cfg.text_length, output_type="pil")
    torch.save({"sink": sink, "regs": regs}, QK_DIR / f"qk_p{prompt_id}_s{gseed}.pt")
    for (bt, bid), p in orig.items():
        tr.single_transformer_blocks[bid].attn.set_processor(p)
    print("saved", QK_DIR / f"qk_p{prompt_id}_s{gseed}.pt")
    print("NOTE: captures the LAST denoising step (dict overwritten each step).")


def analyze_qk():
    import torch.nn.functional as F
    rows = []
    for f in sorted(QK_DIR.glob("qk_*.pt")):
        d = torch.load(f, weights_only=False)
        for tag, s in d["sink"].items():
            reg = d["regs"].get(tag)
            if reg is None or reg.numel() == 0:
                continue
            qm = F.normalize(s["q_mean"], dim=-1)                     # [H, Dh]
            ks = F.normalize(s["keys"], dim=-1)                       # [H, N, Dh]
            cos = torch.einsum("hd,hnd->hn", qm, ks)                  # [H, N]
            r0 = int(reg[0])
            rank = (cos > cos[:, r0:r0 + 1]).sum(dim=1) + 1           # rank of register key per head
            rows.append(dict(file=f.name, block=tag,
                             mean_cos_register=float(cos[:, r0].mean()),
                             mean_cos_all=float(cos.mean()),
                             median_rank_of_register_key=float(rank.float().median()),
                             frac_heads_register_top10=float((rank <= 10).float().mean())))
    df = pd.DataFrame(rows)
    print("Q1d QK GEOMETRY — alignment of the register KEY with each head's mean QUERY:")
    print(df.round(3).to_string(index=False))
    print("""
   median rank near 1 + frac_top10 high -> the register's key direction is what heads' queries
   point at: attention prefers v* because Wk maps v* (post-LN) onto the heads' query consensus.
   rank ~ N/2 -> preference is NOT first-order q·k geometry; look at RoPE phase or per-head offsets.""")
    df.to_csv(QK_DIR / "qk_alignment.csv", index=False)


# ---- run ----
for p in range(len(int_cfg.base.prompts)):
     run_qk_capture(int_cfg.base, prompt_id=p, gseed=0)
analyze_qk()

In [ ]:
# ================================================================
# Cell 27: final mechanistic checks (three parts)
#   27a  Sign check: is x[register, 154] one sign everywhere?     (capA files, no GPU)
#   27b  Birth operator scatter at dual18: Δ154 vs existing 154   (capB files, no GPU)
#   27c  Maintenance attribution in single blocks: attn vs MLP    (one fresh 6-gen run)
# Requires: Cells 1-7, 9 definitions; VSTAR_DIR/RUNB_DIR from Cells 23-24.
# ================================================================

CH = 154   # the register channel

# ---------------------------------------------------------------
# 27a: sign check across all captured register vectors
# ---------------------------------------------------------------
def check_sign():
    vals = []
    for f in sorted(VSTAR_DIR.glob("capA_*.pt")):
        cap = torch.load(f, weights_only=False)
        for (bt, bid, st), d in cap["data"].items():
            for i in range(d["register_vecs"].shape[0]):
                vals.append(dict(btype=bt, bid=bid, step=st, x154=float(d["register_vecs"][i, CH]),
                                 x1446=float(d["register_vecs"][i, 1446])))
    df = pd.DataFrame(vals)
    for c in ["x154", "x1446"]:
        pos = (df[c] > 0).mean()
        print(f"27a SIGN CHECK {c}: {len(df)} register vectors — positive fraction = {pos:.3f} "
              f"(range [{df[c].min():.0f}, {df[c].max():.0f}])")
    print("   1.000 or 0.000 -> single signed population (matches the reference sign-consistency for 154).")
    print("   Mixed -> two signed subpopulations; the sign-alignment step in Cell 23 hid this — report it.")
    print(df.groupby(["btype", "bid"])["x154"].agg(["mean", "min", "max"]).round(0).to_string())
    return df

sign_df = check_sign()


# ---------------------------------------------------------------
# 27b: the birth operator, delta154 vs existing 154 at dual 17/18
# ---------------------------------------------------------------
def birth_scatter(layers=(17, 18)):
    """Pool all tokens over captures/steps. For each dual layer L:
       x = |proj| entering L (= post of L-1); Δmlp = post(L) − postAttn(L); Δattn = postAttn(L) − post(L−1).
       Winner-take-all signature: Δmlp negative/flat below a knee, explosive above it."""
    pts = {L: {"x": [], "dmlp": [], "dattn": []} for L in layers}
    for f in sorted(RUNB_DIR.glob("capB_*.pt")):
        cap = torch.load(f, weights_only=False)
        steps = sorted({k[1] for k in cap["post"].keys()})
        for st in steps:
            for L in layers:
                if (L, st) not in cap["postattn"] or (L - 1, st) not in cap["post"]:
                    continue
                pre, pa, po = cap["post"][(L - 1, st)][:, 1], cap["postattn"][(L, st)][:, 1], cap["post"][(L, st)][:, 1]
                s = torch.sign(pre + 1e-9)
                pts[L]["x"].append((pre * s))           # signed-aligned existing component
                pts[L]["dattn"].append((pa - pre) * s)
                pts[L]["dmlp"].append((po - pa) * s)
    fig, axes = plt.subplots(1, len(layers), figsize=(7 * len(layers), 5), squeeze=False)
    for ax, L in zip(axes[0], layers):
        x = torch.cat(pts[L]["x"]); dm = torch.cat(pts[L]["dmlp"]); da = torch.cat(pts[L]["dattn"])
        ax.scatter(x.numpy(), da.numpy(), s=2, alpha=0.15, label="attention write", color="tab:orange")
        ax.scatter(x.numpy(), dm.numpy(), s=2, alpha=0.15, label="MLP write", color="tab:blue")
        ax.set_xscale("symlog", linthresh=100); ax.set_yscale("symlog", linthresh=100)
        ax.axhline(0, color="gray", lw=0.6)
        ax.set_xlabel("existing |x_154| entering the layer"); ax.set_ylabel("Δ x_154 (sign-aligned)")
        ax.set_title(f"dual layer {L}: the birth operator")
        ax.legend(markerscale=6)
        # knee summary: median Δmlp in log-spaced x bins
        edges = torch.tensor([0, 200, 500, 1000, 2000, 4000, 8000, 1e9])
        med = [float(dm[(x >= a) & (x < b)].median()) if ((x >= a) & (x < b)).any() else float("nan")
               for a, b in zip(edges[:-1], edges[1:])]
        print(f"27b layer {L}: median MLP Δ154 by existing-x bin "
              f"{[f'{a:.0f}-{b:.0f}:{m:.0f}' for a, b, m in zip(edges[:-1], edges[1:], med)]}")
    fig.tight_layout(); fig.savefig(VSTAR_DIR / "fig_birth_operator_scatter.png", dpi=180); plt.show()

birth_scatter()


# ---------------------------------------------------------------
# 27c: who maintains the register in SINGLE blocks, attention or MLP?
# ---------------------------------------------------------------
# FluxSingleTransformerBlock: hidden += gate * proj_out( cat[attn_out, mlp_hidden] ).
# proj_out.weight row CH therefore splits the channel-154 write EXACTLY:
#   write154 = gate154 * ( attn_out @ W[CH,:D]  +  mlp_hidden @ W[CH,D:]  +  b[CH] )
# We hook proj_out's input, slice its own weight, and log the two terms for register tokens.

SINGLE_ATTRIB_BLOCKS = [0, 2, 4, 6, 8, 10, 12, 14, 16]
ATTRIB_DIR = VSTAR_DIR / "single_attrib"; ATTRIB_DIR.mkdir(exist_ok=True)


def run_single_attrib(cfg: "ExperimentConfig") -> None:
    from diffusers import FluxPipeline
    dtype = {"float16": torch.float16, "bfloat16": torch.bfloat16, "float32": torch.float32}[cfg.dtype]
    pipe = FluxPipeline.from_pretrained(cfg.model_name, torch_dtype=dtype).to(cfg.device)
    pipe.set_progress_bar_config(disable=True)
    tr = pipe.transformer
    D = tr.config.num_attention_heads * tr.config.attention_head_dim   # 3072
    state = {"step": -1, "regs": {}, "rows": []}

    def make_block_pre(bid, text_len):
        def pre(mod, args, kwargs):
            x = kwargs.get("hidden_states", args[0] if args else None)
            n = x[0].float().norm(dim=-1)
            r = torch.nonzero(n > 3.0 * n.median()).flatten()
            r = r[torch.argsort(n[r], descending=True)][:8]
            state["regs"][bid] = r
            return None
        return pre

    def make_projout_pre(bid, blk):
        w = blk.proj_out.weight[CH].detach()            # [D + mlp_dim]
        b = float(blk.proj_out.bias[CH].detach()) if blk.proj_out.bias is not None else 0.0

        def pre(mod, args, kwargs):
            cat = args[0] if args else kwargs.get("input", None)
            if cat is None:
                return None
            reg = state["regs"].get(bid)
            if reg is None or reg.numel() == 0:
                return None
            c = cat[0]                                   # [S, D+mlp]; single blocks: S = text+img
            img_start = c.shape[0] - 1024 if c.shape[0] > 1024 else 0   # image tokens are the tail
            for t in reg.tolist():
                row = c[img_start + t].float()
                a154 = float(row[:D] @ w[:D].float())
                m154 = float(row[D:] @ w[D:].float())
                state["rows"].append(dict(step=state["step"], block=bid, token=int(t),
                                          attn_write154=a154, mlp_write154=m154, bias154=b))
            return None
        return pre

    handles = [tr.register_forward_pre_hook(
        lambda m, a, k: state.__setitem__("step", state["step"] + 1), with_kwargs=True)]
    for bid in SINGLE_ATTRIB_BLOCKS:
        blk = tr.single_transformer_blocks[bid]
        handles.append(blk.register_forward_pre_hook(make_block_pre(bid, cfg.text_length), with_kwargs=True))
        handles.append(blk.proj_out.register_forward_pre_hook(make_projout_pre(bid, blk), with_kwargs=True))

    try:
        for pid, prompt in enumerate(cfg.prompts):
            for gseed in cfg.seeds:
                fp = ATTRIB_DIR / f"attrib_p{pid}_s{gseed}.csv"
                if fp.exists():
                    print(f"skip p{pid}s{gseed}"); continue
                state["step"], state["rows"], state["regs"] = -1, [], {}
                gen = torch.Generator(device=cfg.device).manual_seed(gseed)
                pipe(prompt=prompt, height=cfg.height, width=cfg.width,
                     num_inference_steps=cfg.num_inference_steps, guidance_scale=cfg.guidance_scale,
                     generator=gen, max_sequence_length=cfg.text_length, output_type="pil")
                pd.DataFrame(state["rows"]).to_csv(fp, index=False)
                print(f"saved {fp.name}")
    finally:
        for h in handles:
            h.remove()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def analyze_single_attrib():
    df = pd.concat([pd.read_csv(f) for f in sorted(ATTRIB_DIR.glob("attrib_*.csv"))], ignore_index=True)
    df["total"] = df["attn_write154"] + df["mlp_write154"] + df["bias154"]
    df["mlp_share"] = df["mlp_write154"].abs() / (df["mlp_write154"].abs() + df["attn_write154"].abs() + 1e-9)
    g = df.groupby("block").agg(mean_attn=("attn_write154", "mean"), mean_mlp=("mlp_write154", "mean"),
                                mean_total=("total", "mean"), mlp_share=("mlp_share", "mean"),
                                n=("token", "size"))
    print("27c SINGLE-ZONE MAINTENANCE — channel-154 write onto register tokens, attn vs MLP path:")
    print(g.round(2).to_string())
    print("""   mlp_share ~1 and mean_mlp >> mean_attn -> the refill is the MLP path (predicted).
   NOTE: writes are pre-gate; gate scales both paths identically, so the SHARE is exact.
   Signed means matter: positive mean along the register's sign = amplification.""")
    plt.figure(figsize=(9, 4))
    plt.plot(g.index, g["mean_mlp"], marker="o", label="MLP write (ch 154)")
    plt.plot(g.index, g["mean_attn"], marker="o", label="attention write (ch 154)")
    plt.axhline(0, color="gray", lw=0.6)
    plt.xlabel("single block"); plt.ylabel("mean write to ch 154 on register tokens (pre-gate)")
    plt.title("Who maintains the register? attention vs MLP path inside single blocks")
    plt.legend(); plt.tight_layout()
    plt.savefig(VSTAR_DIR / "fig_single_maintenance_attribution.png", dpi=180); plt.show()

run_single_attrib(int_cfg.base)
analyze_single_attrib()

In [ ]:
# ================================================================
# Cell 28: what selects the ~3 register winners from the ~70 candidates?
# ================================================================
# Three mutually-distinguishing hypotheses, all answerable from files on disk:
#   H-pos    : winners are the same GRID POSITIONS regardless of prompt/seed
#   H-content: winners sit on low-detail image regions (flat background/sky)
#   H-noise  : which candidate wins is set by tiny fluctuations -> changes across SEEDS
#              for the SAME prompt (rich-get-richer on an unstable pool)
#
# Decision logic:
#   same positions across BOTH prompt and seed        -> H-pos (architectural/RoPE)
#   same across seed, differ across prompt             -> H-content
#   differ across seed (same prompt)                   -> H-noise
#   low seed-overlap + high edge-enrichment            -> H-noise on a positionally-biased pool
#
# Inputs (already on Drive):
#   RUNB_DIR/capB_*.pt              register token ids per (prompt,seed,step) at register_layer
#   images/<baseline>/promptP_seedS.png   for the content/gradient test
# Requires: Cells 23-24 ran (RUNB_DIR, VSTAR_DIR defined). No GPU.

import numpy as np
from itertools import combinations

GRID = 32                    # 512/16
REGISTER_LAYER = 19          # single_block0 == layer 19; where we define the register set
SEL_OUT = VSTAR_DIR / "selection"; SEL_OUT.mkdir(exist_ok=True)


def _register_ids_from_capB(register_layer=REGISTER_LAYER, ratio=3.0):
    """Recover register token ids per (prompt,seed,step) from the birth-capture norms."""
    recs = []
    for f in sorted(RUNB_DIR.glob("capB_*.pt")):
        cap = torch.load(f, weights_only=False)
        steps = sorted({k[1] for k in cap["post"].keys()})
        for st in steps:
            norms = cap["post"][(register_layer, st)][:, 0]      # [N]
            reg = torch.nonzero(norms > ratio * norms.median()).flatten().tolist()
            recs.append(dict(prompt_id=cap["prompt_id"], seed=cap["seed"], step=st,
                             ids=tuple(sorted(reg))))
    return pd.DataFrame(recs)


def _jaccard(a, b):
    a, b = set(a), set(b)
    return len(a & b) / len(a | b) if (a | b) else float("nan")


def analyze_selection():
    df = _register_ids_from_capB()
    if df.empty:
        print("No capB files found."); return
    # Use the final denoising step (registers most established).
    laststep = df["step"].max()
    dl = df[df["step"] == laststep]

    # ---- H-pos vs H-content vs H-noise via Jaccard structure ----
    same_prompt_diff_seed, diff_prompt = [], []
    for (p, s1), (p2, s2) in combinations(dl.set_index(["prompt_id", "seed"]).index, 2):
        r1 = dl[(dl.prompt_id == p) & (dl.seed == s1)]["ids"].iloc[0]
        r2 = dl[(dl.prompt_id == p2) & (dl.seed == s2)]["ids"].iloc[0]
        j = _jaccard(r1, r2)
        if p == p2:
            same_prompt_diff_seed.append(j)
        else:
            diff_prompt.append(j)
    sp = np.nanmean(same_prompt_diff_seed) if same_prompt_diff_seed else float("nan")
    dp = np.nanmean(diff_prompt) if diff_prompt else float("nan")

    expected = (3*3/(GRID*GRID)) / (2*3 - (3*3/(GRID*GRID)))
    sizes = dl["ids"].apply(len)
    print("28 SELECTION ANALYSIS (register token-id overlap, final step):")
    print(f"   register-set sizes: min={sizes.min()}, median={sizes.median():.0f}, max={sizes.max()}   "
          f"(if ~0-1, a 0.000 Jaccard is an artifact, not a finding)")
    print(f"   same prompt / different seed : mean Jaccard = {sp:.3f}   (n={len(same_prompt_diff_seed)})")
    print(f"   different prompt             : mean Jaccard = {dp:.3f}   (n={len(diff_prompt)})")
    print(f"   chance Jaccard for ~3 of {GRID*GRID} tokens ≈ {expected:.4f}")
    print("""   Interpretation:
     both high (>>chance)                 -> H-POS: same grid positions regardless of content
     same-seed high, diff-prompt low      -> H-CONTENT: winners track image content
     both near chance / seed-unstable     -> H-NOISE: winner identity set by fluctuations""")

    # explicit seed-stability: for each prompt, overlap across its seeds
    print("\n   Per-prompt seed stability (Jaccard across seeds, same prompt):")
    for p, g in dl.groupby("prompt_id"):
        ids = list(g["ids"])
        js = [_jaccard(a, b) for a, b in combinations(ids, 2)] if len(ids) > 1 else [float("nan")]
        print(f"     prompt {p}: mean Jaccard across seeds = {np.nanmean(js):.3f}  (seeds={len(ids)})")

    # ---- position map: where on the grid do winners land, pooled ----
    heat = np.zeros((GRID, GRID))
    for ids in dl["ids"]:
        for t in ids:
            heat[t // GRID, t % GRID] += 1
    edge = sum(heat[r, c] for r in range(GRID) for c in range(GRID)
               if r == 0 or c == 0 or r == GRID-1 or c == GRID-1)
    total = heat.sum()
    edge_frac_uniform = (4*GRID - 4) / (GRID*GRID)
    print(f"\n   Edge occupancy: {edge/total:.3f} of winners on the border "
          f"(uniform baseline {edge_frac_uniform:.3f}, enrichment {(edge/total)/edge_frac_uniform:.1f}x)")

    # ---- content test: do winners sit on low-gradient image patches? ----
    from PIL import Image
    base_dir = None
    for cand in ["sham", "clean"]:
        d = Path(int_cfg.output_dir) / "images" / cand
        if d.exists():
            base_dir = d; break
    content_rows = []
    if base_dir is not None:
        for _, r in dl.iterrows():
            fp = base_dir / f"prompt{r.prompt_id}_seed{r.seed}.png"
            if not fp.exists():
                continue
            img = np.array(Image.open(fp).convert("L").resize((GRID, GRID))).astype(float)
            gx, gy = np.gradient(img)
            grad = np.hypot(gx, gy)                       # [GRID,GRID] local detail
            gpct = grad / (grad.max() + 1e-9)
            for t in r.ids:
                content_rows.append(gpct[t // GRID, t % GRID])
        if content_rows:
            cr = np.array(content_rows)
            print(f"\n   Content test: winner-patch gradient percentile "
                  f"mean={cr.mean():.3f}, median={np.median(cr):.3f}")
            print("     << 0.5 -> winners sit on LOW-detail regions (H-content corroborated)")
            print("     ~ 0.5 -> no content preference")

    # ---- figure ----
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))
    im = ax[0].imshow(heat, cmap="inferno")
    ax[0].set_title(f"Register winner positions (pooled)\nedge enrichment "
                    f"{(edge/total)/edge_frac_uniform:.1f}x")
    fig.colorbar(im, ax=ax[0], fraction=0.046)
    ax[1].bar(["same prompt\ndiff seed", "diff prompt", "chance"], [sp, dp, expected],
              color=["tab:green", "tab:purple", "gray"])
    ax[1].set_ylabel("mean Jaccard"); ax[1].set_title("Winner-set overlap")
    fig.tight_layout(); fig.savefig(SEL_OUT / "fig_selection.png", dpi=180); plt.show()

    enrichment = (edge / total) / edge_frac_uniform if total > 0 else float("nan")
    if sp < 0.25:
        verdict = ("H-NOISE with POSITIONAL BIAS (border-enriched pool, fluctuation-picked winners)"
                   if enrichment > 2 else "H-NOISE (fluctuation-set, no positional bias)")
    elif sp > 0.4 and dp > 0.4:
        verdict = "H-POS (positional/architectural — fixed register slots)"
    elif sp > 0.4 and dp < 0.25:
        verdict = "H-CONTENT (winners track image content)"
    else:
        verdict = "MIXED — inspect the per-prompt table"
    print(f"\n   >>> VERDICT: {verdict}")
    print(f"       (seed-overlap {sp:.3f}, prompt-overlap {dp:.3f}, edge-enrichment {enrichment:.1f}x)")


analyze_selection()

In [ ]:
# Did the transplanted register FLATTEN its recipient patch?  (register -> flatness test)
import numpy as np, pandas as pd
from PIL import Image
from pathlib import Path
GRID = 32
ROOT = Path(int_cfg.output_dir)
dl = pd.read_csv(ROOT / "dose_log.csv")
dl = dl[(dl["condition"] == "transplant_a12") & (dl["recipient_token"] >= 0)]

def patch_grad(img_path, tok):
    img = np.array(Image.open(img_path).convert("L").resize((GRID, GRID))).astype(float)
    gx, gy = np.gradient(img); g = np.hypot(gx, gy)
    return g[tok // GRID, tok % GRID] / (g.max() + 1e-9)

rows = []
for (p, s), g in dl.groupby(["prompt_id", "seed"]):
    tok = int(g["recipient_token"].iloc[-1])           # final-step recipient
    tp = ROOT / "images" / "transplant_a12" / f"prompt{p}_seed{s}.png"
    sh = ROOT / "images" / "sham" / f"prompt{p}_seed{s}.png"
    if tp.exists() and sh.exists():
        rows.append(dict(prompt=p, seed=s, token=tok,
                         grad_transplant=patch_grad(tp, tok), grad_sham=patch_grad(sh, tok)))
r = pd.DataFrame(rows)
r["delta"] = r["grad_transplant"] - r["grad_sham"]
print(r.round(3).to_string(index=False))
print(f"\nmean Δ detail at recipient patch = {r['delta'].mean():+.3f}  "
      f"({(r['delta'] < 0).sum()}/{len(r)} pairs flatter under transplant)")
print("consistently negative -> registers CAUSE local flatness (occupancy suppresses detail);")
print("~zero -> flat regions attract registers (selection-by-content).")